# Mapping Disordered Gaming — R2 computational analysis (public source)

**Companion code for** “Mapping Disordered Gaming: Semantic Patterns and Domain-Specific Language Model Classification”. **Manuscript DHJ-26-2664.R2.** This is an output-free, source-only release of the research computational pipeline, not a package of individual participant or post records.

## Study design and model evaluation

- Candidate posts are coded against the primary self-disclosure rule. Final aggregate labels: **58 Include / 300 Exclude / 7 Uncertain** from 365 screened candidates. Uncertain records are withheld from binary classification.
- The stricter **18-post subset** is a nested sensitivity analysis; its exact scope and limitations are documented in the manuscript and Supplementary Material.
- RQ1 uses pretrained MentalBERT text representations, condition-specific PCA and multiple-comparison controls.
- RQ2 compares baseline rules, TF-IDF and fine-tuned MentalBERT using author-grouped out-of-fold evaluation, raw and keyword-masked text, multiple comparison sets, and fixed training seeds **42, 123, 2026**.
- **Primary fine-tuned ROC-AUC** is the arithmetic mean of the three seed-specific ROC-AUC values. An AUC computed from averaged seed predictions is a **separate ensemble-based** statistic, used for some additional paired comparisons; the two must not be conflated.
- External DDO/WoW model-consensus selection is exploratory candidate prioritisation followed by independent researcher assessment; it does not estimate diagnostic accuracy, corpus-wide sensitivity or disorder prevalence.

## Run requirements and privacy

This notebook was designed for Google Colab with GPU and locally authorised access to research inputs. It **cannot be run end to end using only this public download**, because full post text, coded reviews, post IDs, author-group assignments and some trained artifacts are private or subject to access restrictions. An additional local `strict18_selection_PRIVATE.json` config holds the frozen sensitivity-subset audit IDs, deliberately omitted from the public release. Input filenames in the source are local placeholders for those protected materials; no source posts or private review exports are packaged here.

No outputs, execution logs, tokens, row-level model predictions, historical manual-review IDs or review sheets are included. The original internal notebook remains unchanged and is **not intended for public upload**. The source was structurally checked and Python code compiled; no complete GPU retraining was run as part of this publication preparation.

**Code section order:** setup → label/corpus checks → lexical baselines → pretrained RQ1/RQ2 → fine-tuning → external candidate selection → non-identifying diagnostic figures. Some private recovery/export utilities from the working notebook are intentionally absent; their removal does not modify model-training functions or reported aggregate results.


In [ ]:

# CELL 00 — Hugging Face 4.x compatibility + Colab GPU preflight.
# This project uses the Transformers 4.x API (as previously tested).
# DO NOT reinstall PyTorch: keep the GPU-enabled CUDA build provided by Colab.

import importlib.util
import importlib.metadata as md
import subprocess
import sys
from packaging.version import Version

# Versions known to have passed the earlier project's import preflight.
# Sentence Transformers 5.7.0 supports Transformers >=4.41,<6, so keep it.
PINNED = {
    'transformers': '4.57.6',
    'huggingface-hub': '0.36.2',
    'tokenizers': '0.22.1',
}

OTHER_REQUIRED = {
    'numpy': ('numpy', 'numpy>=1.24,<3'),
    'pandas': ('pandas', 'pandas>=2,<3'),
    'scipy': ('scipy', 'scipy>=1.10,<2'),
    'sklearn': ('scikit-learn', 'scikit-learn>=1.5,<2'),
    'statsmodels': ('statsmodels', 'statsmodels>=0.14,<1'),
    'sentence_transformers': (
        'sentence-transformers',
        'sentence-transformers>=3,<6'
    ),
    'openpyxl': ('openpyxl', 'openpyxl'),
    'sentencepiece': ('sentencepiece', 'sentencepiece'),
    'ftfy': ('ftfy', 'ftfy'),
    'joblib': ('joblib', 'joblib'),
    'matplotlib': ('matplotlib', 'matplotlib'),
    'tqdm': ('tqdm', 'tqdm'),
}

def installed_version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

# Reinstall only the incompatible Hugging Face packages.
# --no-deps prevents pip from replacing torch+CUDA.

to_install = [
    f'{name}=={version}'
    for name, version in PINNED.items()
    if installed_version(name) != version
]

# Install missing packages only.
for module, (package, spec) in OTHER_REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        to_install.append(spec)

if to_install:
    print('Fixing only incompatible/missing project dependencies:')

    for item in to_install:
        print('  ', item)

    cmd = [
        sys.executable, '-m', 'pip', 'install',
        '-q', '--no-cache-dir', '--no-deps',
        *to_install
    ]

    subprocess.check_call(cmd)

    print(
        '\nPackages installed. IMPORTANT: '
        'Runtime -> Restart session, then rerun Cell 00.'
    )

    raise SystemExit(
        'Restart the Colab session once, '
        'then rerun cells 00 and 01.'
    )

# Verify environment.
checks = [
    'torch',
    'transformers',
    'sentence-transformers',
    'huggingface-hub',
    'tokenizers',
    'scikit-learn',
    'numpy',
    'pandas',
    'scipy',
    'ftfy',
]

for name in checks:
    print(f'{name}: {installed_version(name)}')

assert installed_version('torch') is not None, \
    'PyTorch not installed.'

assert installed_version('transformers') == PINNED['transformers']
assert installed_version('huggingface-hub') == PINNED['huggingface-hub']
assert installed_version('tokenizers') == PINNED['tokenizers']

assert (
    3 <= Version(
        installed_version('sentence-transformers')
    ).major <= 5
)

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

from sentence_transformers import SentenceTransformer

print('\nRequired imports: PASS')
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU NOT ACTIVE. Colab -> Runtime -> '
        'Change runtime type -> T4 GPU.'
    )

print('GPU:', torch.cuda.get_device_name(0))
print('PASS: Hugging Face 4.x environment + GPU ready for the R2 notebook.')


In [ ]:

# 01 — R2 58/18 project configuration.
# Keep all previous R1/R2 projects unchanged.

import os, re, gc, json, hashlib, random, platform, sys
import html, unicodedata, math, warnings

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import scipy.stats as st

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE = Path.cwd()

ROOT = DRIVE / 'MMORPG_DigitalHealth_R2_58_18_v6_FINAL_NO415'

PREVIOUS_INPUTS = [
    DRIVE / 'MMORPG_DigitalHealth_R2_58_18_v5_1to2' / '00_inputs',
    DRIVE / 'MMORPG_DigitalHealth_R2_58_18_v4' / '00_inputs'
]

INPUT = ROOT / '00_inputs'
AUDIT = ROOT / '01_audits'
DATA = ROOT / '02_datasets'
RQ1 = ROOT / '03_rq1'
BASE = ROOT / '04_baselines'
FT = ROOT / '05_finetune'
EXTERNAL = ROOT / '06_external_private'
MANUSCRIPT = ROOT / '07_manuscript_review_gates'

for p in (
    INPUT, AUDIT, DATA, RQ1,
    BASE, FT, EXTERNAL, MANUSCRIPT
):
    p.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

def existing_input(filename):
    current = INPUT / filename

    if current.exists():
        return current

    for folder in PREVIOUS_INPUTS:
        previous = folder / filename
        if previous.exists():
            return previous

    return current


LABEL_XLSX = existing_input(
    'MMORPG_365_Three_Reviews_Final_PLAIN_Corrected_Criteria_EN.xlsx'
)

MASTER_CSV = (
    DRIVE / 'Reddit_Analysis_Results'
    / 'mmorpg_posts_FINAL_CLEAN.csv'
)


# ------------------------------------------------------------
# PRETRAINED EMBEDDING CACHE
# ------------------------------------------------------------

OLD_EMBED_CACHE = (
    DRIVE / 'MMORPG_DigitalHealth_R2_56'
    / '02_corpus_preparation'
    / 'MentalBERT_512_Resumable_PINNED_24809aa'
)

NEW_EMBED_CACHE = (
    ROOT / '03_rq1'
    / 'MentalBERT_512_Resumable_PINNED_24809aa'
)


# ------------------------------------------------------------
# ANALYSIS OPTIONS
# ------------------------------------------------------------

RUN_RQ1_REPEATED_RANDOM = True
RUN_FINE_TUNING = True

# External runs will be activated separately.
RUN_EXTERNAL = False
RUN_WOW = False


# ------------------------------------------------------------
# OPTIONAL VALIDATION AND EXTERNAL FILES
# ------------------------------------------------------------

HIGHSPEC_IDS_CSV = existing_input(
    'high_specificity_18_ids_final_verified.csv'
)

OUTSIDE_KEYWORD_AUDIT_CSV = existing_input(
    'outside_keyword_300_new_rule_independent_review.csv'
)

DDO_CSV = existing_input(
    'DDO_eligible_948_PRIVATE.csv'
)

OLD_DDO_IDS = existing_input(
    'DDO_historical_36_candidate_ids_PRIVATE.csv'
)

OLD_WOW_IDS = existing_input(
    'WoW_historical_5243_candidate_ids_PRIVATE.csv'
)

WOW_CSV = existing_input(
    'WoW_eligible_PRIVATE.csv'
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

MODEL_NAME = 'mental/mental-bert-base-uncased'

MODEL_REVISION = (
    '24809aa822c76639760d0d934742d1b42f89942f'
)

MAX_TOKENS = 512

SEEDS = [42, 123, 2026]
CV_SEED = 42

N_FOLDS = 5
N_BOOT = 5000


# ------------------------------------------------------------
# FINAL HUMAN REVIEW COUNTS
# ------------------------------------------------------------

N_POS_EXPECTED = 58
N_NEG_EXPECTED = 300
N_UNCERTAIN_EXPECTED = 7

STRICT_V2_CONFIG = INPUT / "strict18_selection_PRIVATE.json"
assert STRICT_V2_CONFIG.is_file(), (
    "Authorised strict-18 selection configuration missing: "
    + str(STRICT_V2_CONFIG)
)
strict_cfg = json.loads(STRICT_V2_CONFIG.read_text(encoding="utf-8"))
EXPECTED_STRICT_V2_AUDIT_IDS = list(strict_cfg["audit_ids"])
STRICT_V2_SWAPS = dict(strict_cfg["swaps"])
assert len(EXPECTED_STRICT_V2_AUDIT_IDS) == 18
assert len(set(EXPECTED_STRICT_V2_AUDIT_IDS)) == 18
assert len(STRICT_V2_SWAPS) == 2


# ------------------------------------------------------------
# FINE-TUNING
# ------------------------------------------------------------

MAX_FT_EPOCHS = 2
FT_BATCH = 4
FT_ACCUM = 2
FT_LR = 2e-5

FT_TRAINING_PROTOCOL = (
    'r1_constant_lr_no_scheduler_no_clip_amp'
)

# Primary 1:2:
# 58 positive vs 116 negative
# 18 strict positive vs 36 negative
#
# No 415-control experiments.
# All 300 reviewed excludes are retained for
# explicitly labelled hard-negative sensitivity analyses.


# ------------------------------------------------------------
# KEYWORD MASKING
# ------------------------------------------------------------

STRICT5 = re.compile(
    r'\b(?:addict|addicted|addiction|addictive|hooked)\b',
    re.I
)

BROAD = re.compile(
    r'\b(?:addict\w*|hook\w*|'
    r'can[\u2019\x27]?t\s+stop(?:\s+playing)?|'
    r'cannot\s+stop(?:\s+playing)?|'
    r'couldn[\u2019\x27]?t\s+stop(?:\s+playing)?)\b',
    re.I
)

def mask_text(s):
    return BROAD.sub(' ', str(s))


# ------------------------------------------------------------
# OUTPUT HELPERS
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, 'rb') as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b''
        ):
            h.update(chunk)

    return h.hexdigest()


def save_json(path, obj):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str
        ),
        encoding='utf-8'
    )


def atomic_csv(frame, path):
    tmp = path.with_name(path.name + '.partial')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


# ------------------------------------------------------------
# VERIFY INPUTS
# ------------------------------------------------------------

print('ROOT:', ROOT)

print(
    'Final adjudication workbook:',
    LABEL_XLSX,
    'exists:',
    LABEL_XLSX.exists()
)

print(
    'Canonical master:',
    MASTER_CSV,
    'exists:',
    MASTER_CSV.exists()
)

print('GPU:', torch.cuda.is_available())
print('External planned:', RUN_EXTERNAL)
print('WoW planned:', RUN_WOW)

print(
    'Required label exists:',
    LABEL_XLSX.exists(),
    '| older input folders:',
    [p.exists() for p in PREVIOUS_INPUTS]
)

print(
    'For current tasks: source master exists:',
    MASTER_CSV.exists()
)

print(
    'Optional 948 DDO full corpus:',
    DDO_CSV.exists(),
    '| optional WoW full corpus:',
    WOW_CSV.exists()
)


# ------------------------------------------------------------
# CONFIGURATION ASSERTIONS
# ------------------------------------------------------------

assert MAX_TOKENS == 512
assert SEEDS == [42, 123, 2026]
assert N_FOLDS == 5

assert LABEL_XLSX.exists(), 'Final label workbook missing.'
assert MASTER_CSV.exists(), 'Canonical master CSV missing.'

if RUN_EXTERNAL and not DDO_CSV.exists():
    raise FileNotFoundError(
        'RUN_EXTERNAL=True requires the ENTIRE '
        '948-row DDO CSV.'
    )

if RUN_EXTERNAL and RUN_WOW and not WOW_CSV.exists():
    raise FileNotFoundError(
        'RUN_WOW=True requires the ENTIRE '
        '265009-row WoW CSV.'
    )


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

random.seed(CV_SEED)
np.random.seed(CV_SEED)
torch.manual_seed(CV_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CV_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from importlib.metadata import version as package_version

ENVIRONMENT = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'transformers': package_version('transformers'),
    'sentence_transformers': package_version('sentence-transformers'),
    'huggingface_hub': package_version('huggingface-hub'),
    'scikit_learn': package_version('scikit-learn'),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'cuda_active': torch.cuda.is_available(),
    'cuda_device': (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    'model': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'max_tokens': MAX_TOKENS,
    'seeds': SEEDS,
    'cv_folds': N_FOLDS,
    'fine_tuning_protocol': FT_TRAINING_PROTOCOL,
}

save_json(
    AUDIT / 'environment_preflight.json',
    ENVIRONMENT
)

print(
    'Environment recorded:',
    AUDIT / 'environment_preflight.json'
)

if RUN_FINE_TUNING and not torch.cuda.is_available():
    print(
        'ACTION REQUIRED: Enable GPU before MentalBERT training.'
    )

if LABEL_XLSX.exists() and MASTER_CSV.exists():
    print('SOURCE INPUT CHECK: PASS — no re-upload needed.')


### Model access (one-time in your own Colab session)

The pinned MentalBERT model on Hugging Face may require accepting model access terms in your personal account. The following cell prompts securely for authentication only if no token is already available; **never enter tokens in notebook source**. It does not write a token into the repository.


In [ ]:
from huggingface_hub import get_token, login, model_info
if not get_token():
    login()  # interactive browser/notebook credential prompt; never paste token into source
model_info(MODEL_NAME, revision=MODEL_REVISION)
print("Pinned MentalBERT model access verified.")


In [ ]:
# 02 — Strict 365-row audit. VAS adjudicates exactly the 35 disagreements.
assert LABEL_XLSX.exists(), ('Copy the supplied final plain Excel into '+str(INPUT))
book=pd.ExcelFile(LABEL_XLSX)
assert 'Candidates 365 primary' in book.sheet_names, f'Unexpected sheet names: {book.sheet_names}'
assert 'Criteria' in book.sheet_names, 'Missing English Criteria sheet'
raw_labels=pd.read_excel(LABEL_XLSX,sheet_name='Candidates 365 primary',dtype=str,keep_default_na=False)
assert 'High-specificity subset' in raw_labels.columns
name_map={
    'Audit ID':'audit_id', 'Post ID':'post_id', 'Title':'title_review',
    'Body':'body_review', 'Pseudonymised author group':'review_author_group',
    'Reviewer 1 - Spiros':'spiros', 'Reviewer 2 - Daniel':'daniel',
    'Reviewer 3 - VAS':'vas', 'Final decision':'final_decision', 'Notes - VAS':'vas_notes',
    'High-specificity subset':'high_specificity_raw'
}
assert set(name_map).issubset(raw_labels.columns),set(name_map)-set(raw_labels.columns)
labels=raw_labels.rename(columns=name_map).copy()
for col in name_map.values():
    labels[col]=labels[col].fillna('').astype(str)
for col in ('audit_id','post_id','review_author_group','spiros','daniel','vas','final_decision'):
    labels[col]=labels[col].str.strip()
for col in ('spiros','daniel','vas','final_decision'):
    labels[col]=labels[col].str.title()

assert len(labels)==365
assert labels.audit_id.nunique()==365 and labels.post_id.nunique()==365
assert labels.post_id.ne('').all() and labels.audit_id.ne('').all()
assert labels.review_author_group.ne('').all()
assert labels.audit_id.str.match(r'^BR\d+$').all()
assert labels.spiros.value_counts().to_dict()=={'Exclude':294,'Include':56,'Uncertain':15}
assert labels.daniel.value_counts().to_dict()=={'Exclude':302,'Include':61,'Uncertain':2}
assert labels.final_decision.value_counts().to_dict()=={'Exclude':300,'Include':58,'Uncertain':7}
allowed={'Include','Exclude','Uncertain'}
assert set(labels.final_decision)<=allowed

disagreement=labels.spiros.ne(labels.daniel)
assert disagreement.sum()==35 and (~disagreement).sum()==330
assert labels.loc[disagreement,'vas'].isin(allowed).all()
assert labels.loc[~disagreement,'vas'].eq('').all()
assert labels.loc[disagreement,'vas_notes'].str.strip().ne('').all()
assert labels.loc[~disagreement,'vas_notes'].str.strip().eq('').all()
expected=labels.spiros.where(~disagreement,labels.vas)
assert expected.eq(labels.final_decision).all(), 'Final label differs from prespecified VAS adjudication.'
assert labels.loc[disagreement,'vas'].value_counts().to_dict()=={'Exclude':17,'Include':11,'Uncertain':7}

labels['label']=labels.final_decision.map({'Include':1,'Exclude':0})
positives_reviewed=labels[labels.final_decision=='Include'].copy()
negatives_reviewed=labels[labels.final_decision=='Exclude'].copy()
uncertain_reviewed=labels[labels.final_decision=='Uncertain'].copy()
assert (len(positives_reviewed),len(negatives_reviewed),len(uncertain_reviewed))==(58,300,7)
# The YES flag is a separate, post-hoc high-specificity v2 selection, NOT a primary label.
strict_markers=labels.high_specificity_raw.fillna('').astype(str).str.strip().str.upper()
assert set(strict_markers.unique()) <= {'','YES'},f'Unexpected YES-marker values: {set(strict_markers)}'
labels['high_specificity_v2']=strict_markers.eq('YES')
strict_rows=labels.loc[labels.high_specificity_v2].copy()
assert len(strict_rows)==18 and strict_rows.post_id.nunique()==18
assert strict_rows.final_decision.eq('Include').all(),'Strict includes nonpositive rows'
assert set(strict_rows.audit_id)==set(EXPECTED_STRICT_V2_AUDIT_IDS),'Strict 18 IDs do not match the two user-approved swaps.'
assert not labels.loc[labels.audit_id.isin(STRICT_V2_SWAPS),'high_specificity_v2'].any()
assert labels.loc[labels.audit_id.isin(STRICT_V2_SWAPS.values()),'high_specificity_v2'].all()
assert strict_rows.review_author_group.nunique()==12, 'Author group count changed; revisit CV assumptions.'
assert (strict_rows.review_author_group.value_counts().max()==7), 'Expected 7/18 posts from the same author group.'
print('STRICT v2 ID + nesting AUDIT PASS (18/18; 12 author groups, largest group=7)')

adjudication_audit=labels[['audit_id','post_id','spiros','daniel','vas','final_decision','vas_notes']].copy()
atomic_csv(adjudication_audit,AUDIT/'365_adjudication_audit_PRIVATE.csv')
atomic_csv(uncertain_reviewed[['audit_id','post_id','final_decision']],AUDIT/'7_UNCERTAIN_WITHHELD_PRIVATE.csv')
atomic_csv(strict_rows[['audit_id','post_id','review_author_group','final_decision']],AUDIT/'strict18_v2_proposed_ids_PRIVATE.csv')

INPUT_SHA=sha256_file(LABEL_XLSX)
label_manifest={'source':LABEL_XLSX.name,'sha256':INPUT_SHA,'n_reviewed':365,
    'final_counts':dict(Counter(labels.final_decision)),'spiros_counts':dict(Counter(labels.spiros)),
    'daniel_counts':dict(Counter(labels.daniel)),'vas_disagreements':35,
    'vas_distribution':dict(Counter(labels.loc[disagreement,'vas'])),
    'rule':'On 330 agreements use common rating; on 35 disagreements use VAS including Uncertain',
    'vas_scope':'Final adjudication decisions recorded for 35 initial disagreements',
    'strict_v2_posthoc':True,'strict_v2_ids':sorted(strict_rows.post_id),
    'strict_v2_human_confirmation':'Strict-specific review status not inferred by computational code',
    'strict_v2_author_groups':int(strict_rows.review_author_group.nunique())}
save_json(AUDIT/'label_manifest.json',label_manifest)
print('LABEL AUDIT PASS:',Counter(labels.final_decision),'VAS decisions:',Counter(labels.loc[disagreement,'vas']))
print('Reviewer agreement matrix:')
print(pd.crosstab(labels.spiros,labels.daniel))

In [ ]:
# 02A — Reviewer agreement before VAS adjudication (descriptive, not external validation).
from sklearn.metrics import cohen_kappa_score
initial_agreement=int(labels.spiros.eq(labels.daniel).sum())
initial_fraction=initial_agreement/len(labels)
initial_kappa=cohen_kappa_score(labels.spiros,labels.daniel,
    labels=['Include','Exclude','Uncertain'])
reliability={'n_reviewed':365,'n_agreed_initially':initial_agreement,
    'percent_agreement':100*initial_fraction,'cohen_kappa_three_category':float(initial_kappa),
    'n_adjudicated_by_VAS':35,'vas_scope':'disagreements only, not blind third rating of all 365'}
save_json(AUDIT/'initial_two_reviewer_reliability.json',reliability)
print('INITIAL CODER AGREEMENT: {:.2f}% | kappa {:.4f}'.format(100*initial_fraction,initial_kappa))
print('VAS adjudicated only the 35 disagreements.')

In [ ]:

# 03 — Canonical 114,489-post master and source-text provenance.
# Component-aware audit: title, body, and verified Excel truncation.
# No changes to reviewer labels or source texts.

assert MASTER_CSV.exists(), f'Missing master corpus: {MASTER_CSV}'

MASTER_SHA = sha256_file(MASTER_CSV)

source = pd.read_csv(
    MASTER_CSV,
    dtype={'post_id': str},
    keep_default_na=False,
    low_memory=False
)

assert {'post_id', 'title', 'selftext', 'author'}.issubset(source.columns)

source['post_id'] = source.post_id.astype(str).str.strip()
assert len(source) == 120541
assert int(source.post_id.eq('').sum()) == 1

source['full_text'] = (
    source.title.astype(str).str.strip()
    + '\n'
    + source.selftext.astype(str).str.strip()
).str.strip()

# ------------------------------------------------------------
# MASTER DEDUPLICATION
# ------------------------------------------------------------

repeated = source[source.post_id.duplicated(keep=False)]

conflicts = repeated.groupby('post_id')['full_text'].nunique()

if int((conflicts > 1).sum()) > 0:
    atomic_csv(
        repeated,
        AUDIT / 'BLOCKING_duplicate_text_conflicts_PRIVATE.csv'
    )
    raise ValueError(
        'Conflicting duplicate post IDs: resolve before modelling.'
    )

master = (
    source.drop_duplicates('post_id', keep='first')
    .reset_index(drop=True)
    .copy()
)

del source, repeated

assert len(master) == 114489
assert master.post_id.is_unique
assert int(master.post_id.eq('').sum()) == 1
assert 120541 - len(master) == 6052

missing = set(labels.post_id) - set(master.post_id)

assert not missing, (
    f'{len(missing)} reviewed posts missing: '
    f'{sorted(missing)[:5]}'
)

# ------------------------------------------------------------
# AUTHOR GROUPING
# ------------------------------------------------------------

def author_group(row):
    value = str(row['author']).strip().lower()

    if value in {
        '', 'none', 'nan', '[deleted]',
        'deleted', '[removed]', 'removed'
    }:
        return 'anonymous__' + str(row['post_id'])

    return 'author_' + hashlib.sha256(
        value.encode('utf-8')
    ).hexdigest()[:24]


master['analysis_group'] = master.apply(
    author_group,
    axis=1
)

master['word_len'] = (
    master.full_text.astype(str).str.split().str.len().astype(int)
)

master['char_len'] = (
    master.full_text.astype(str).str.len().astype(int)
)

master['original_index'] = np.arange(
    len(master), dtype=int
)

# ------------------------------------------------------------
# TEXT PROVENANCE
# ------------------------------------------------------------

audit_text = labels[
    ['audit_id', 'post_id', 'title_review',
     'body_review', 'final_decision']
].merge(
    master[
        ['post_id', 'title', 'selftext',
         'full_text', 'analysis_group']
    ],
    on='post_id',
    validate='one_to_one'
)

def norm_text(x):
    import ftfy

    text = unicodedata.normalize('NFKC', str(x))
    text = html.unescape(text)
    text = ftfy.fix_text(text)

    return re.sub(r'\s+', ' ', text).strip()


audit_text['review_text'] = (
    audit_text.title_review.astype(str)
    + '\n'
    + audit_text.body_review.astype(str)
).str.strip()

audit_text['exact_normalized_match'] = [
    norm_text(a) == norm_text(b)
    for a, b in zip(
        audit_text.review_text,
        audit_text.full_text
    )
]

audit_text['review_chars'] = (
    audit_text.review_text.str.len()
)

audit_text['master_chars'] = (
    audit_text.full_text.str.len()
)

audit_text['relative_char_delta'] = (
    (audit_text.master_chars - audit_text.review_chars).abs()
    / audit_text.master_chars.clip(lower=1)
)

atomic_csv(
    audit_text.loc[~audit_text.exact_normalized_match],
    AUDIT / 'annotation_vs_master_text_mismatches_PRIVATE.csv'
)

text_audit_summary = {
    'exact_or_normalized_matches':
        int(audit_text.exact_normalized_match.sum()),
    'mismatches_needing_provenance_note':
        int((~audit_text.exact_normalized_match).sum()),
    'large_char_deltas_over_2pct':
        int((audit_text.relative_char_delta > .02).sum()),
    'known_excel_truncation_candidate': 'BR0325'
}

save_json(
    AUDIT / 'text_audit_summary.json',
    text_audit_summary
)

# ------------------------------------------------------------
# COMPONENT-AWARE PROVENANCE CHECK
# ------------------------------------------------------------

from difflib import SequenceMatcher

reviewer_strict_post_ids = set(
    strict_rows.post_id.astype(str)
)

audit_text['strict_v2'] = (
    audit_text.post_id.isin(reviewer_strict_post_ids)
)

status_rows = []

for rec in audit_text.itertuples(index=False):

    a_title = norm_text(rec.title_review)
    b_title = norm_text(rec.title)

    a_body = norm_text(rec.body_review)
    b_body = norm_text(rec.selftext)

    title_equal = a_title == b_title
    body_equal = a_body == b_body

    title_similarity = (
        1.0 if title_equal else
        SequenceMatcher(None, a_title, b_title).ratio()
    )

    body_similarity = (
        1.0 if body_equal else
        SequenceMatcher(None, a_body, b_body).ratio()
    )

    body_relative_delta = (
        abs(len(a_body) - len(b_body))
        / max(len(b_body), 1)
    )

    prefix_similarity = float('nan')

    if rec.exact_normalized_match:

        status = 'EXACT'

    elif (
        body_equal
        and not title_equal
        and title_similarity >= 0.80
        and abs(len(a_title) - len(b_title)) <= 25
    ):

        status = 'MINOR_TITLE_VARIANT'

    elif (
        title_equal
        and not body_equal
        and body_similarity >= 0.985
        and body_relative_delta <= 0.03
    ):

        status = 'MINOR_BODY_VARIANT'

    elif (
        str(rec.audit_id) == 'BR0325'
        and str(rec.post_id) == 'stw7uq'
        and str(rec.final_decision) == 'Exclude'
        and title_equal
        and len(a_body) >= 30000
        and len(b_body) > len(a_body)
    ):

        prefix_similarity = SequenceMatcher(
            None,
            a_body,
            b_body[:len(a_body)]
        ).ratio()

        if prefix_similarity >= 0.995:
            status = 'REVIEW_TEXT_TRUNCATED'
        else:
            status = 'BLOCK_UNEXPLAINED_TRUNCATION'

    else:

        status = 'BLOCK_UNEXPLAINED_VARIANT'

    status_rows.append({
        'audit_id': str(rec.audit_id),
        'post_id': str(rec.post_id),
        'final_decision': str(rec.final_decision),
        'strict_v2': bool(rec.strict_v2),
        'status': status,
        'title_equal': title_equal,
        'body_equal': body_equal,
        'title_similarity': round(title_similarity, 5),
        'body_similarity': round(body_similarity, 5),
        'body_relative_delta': round(body_relative_delta, 5),
        'truncated_prefix_similarity': (
            round(prefix_similarity, 5)
            if not np.isnan(prefix_similarity)
            else None
        ),
        'review_chars': int(rec.review_chars),
        'master_chars': int(rec.master_chars)
    })

provenance = pd.DataFrame(status_rows)

assert len(provenance) == 365
assert provenance.post_id.nunique() == 365

atomic_csv(
    provenance,
    AUDIT / 'CELL03_source_text_provenance_classification_PRIVATE.csv'
)

# ------------------------------------------------------------
# GATE SUMMARY
# ------------------------------------------------------------

summary_counts = dict(Counter(provenance.status))

text_audit_summary.update({
    'provenance_gate':
        'Component-aware title/body audit',
    'provenance_categories': summary_counts,
    'unexplained_cases':
        int(provenance.status.str.startswith('BLOCK').sum()),
    'strict18_affected':
        int((
            (provenance.status != 'EXACT')
            & provenance.strict_v2
        ).sum()),
    'positive_affected':
        int((
            (provenance.status != 'EXACT')
            & provenance.final_decision.eq('Include')
        ).sum()),
    'master_text_used_for_training': True
})

save_json(
    AUDIT / 'text_audit_summary.json',
    text_audit_summary
)

print('SOURCE PROVENANCE CATEGORIES:', summary_counts)

print('\nNON-EXACT INCLUDED POSTS:')

print(
    provenance.loc[
        provenance.final_decision.eq('Include')
        & provenance.status.ne('EXACT'),
        [
            'audit_id',
            'post_id',
            'strict_v2',
            'status',
            'title_similarity',
            'body_similarity'
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# BLOCK UNEXPLAINED DIFFERENCES
# ------------------------------------------------------------

blocked = provenance.loc[
    provenance.status.str.startswith('BLOCK')
].copy()

if len(blocked):

    atomic_csv(
        blocked,
        AUDIT / 'BLOCKING_unexplained_source_text_differences_PRIVATE.csv'
    )

    print('\nBLOCKING UNEXPLAINED DIFFERENCES:')

    print(
        blocked[
            [
                'audit_id',
                'post_id',
                'status',
                'title_similarity',
                'body_similarity',
                'truncated_prefix_similarity'
            ]
        ].to_string(index=False)
    )

    raise ValueError(
        f'{len(blocked)} unexplained text differences. '
        'Review the saved blocking audit before continuing.'
    )

# ------------------------------------------------------------
# POSITIVE AND STRICT SAFETY CHECKS
# ------------------------------------------------------------

accepted_minor = {
    'MINOR_TITLE_VARIANT',
    'MINOR_BODY_VARIANT'
}

assert provenance.loc[
    provenance.strict_v2
    & provenance.status.ne('EXACT'),
    'status'
].isin(accepted_minor).all(), (
    'Unresolved strict-case source discrepancy.'
)

assert provenance.loc[
    provenance.final_decision.eq('Include')
    & provenance.status.ne('EXACT'),
    'status'
].isin(accepted_minor).all(), (
    'Unresolved positive-case source discrepancy.'
)

# Save the BR0325 additional text for manual review.
if 'REVIEW_TEXT_TRUNCATED' in summary_counts:

    record = audit_text.loc[
        audit_text.audit_id.eq('BR0325')
    ].iloc[0]

    reviewed_body = norm_text(record.body_review)
    complete_body = norm_text(record.selftext)

    suffix = complete_body[len(reviewed_body):]

    suffix_file = (
        AUDIT / 'BR0325_unseen_master_suffix_PRIVATE.txt'
    )

    suffix_file.write_text(
        suffix,
        encoding='utf-8'
    )

    print(
        '\nCAUTION: BR0325 appears Excel-truncated.'
    )
    print(
        'Full-text suffix saved for manual review:',
        suffix_file
    )
    print(
        'Reviewer Exclude decision must be checked '
        'against this additional text before publication.'
    )

print(
    '\nSOURCE TEXT PROVENANCE PASS:',
    '365 IDs checked;',
    int(provenance.status.eq('EXACT').sum()),
    'exact;',
    int(provenance.status.ne('EXACT').sum()),
    'documented variants;',
    'zero unresolved.'
)

# ------------------------------------------------------------
# FREEZE MASTER MANIFEST
# ------------------------------------------------------------

master_manifest = {
    'master_path': str(MASTER_CSV),
    'master_sha256': MASTER_SHA,
    'n_raw': 120541,
    'n_unique': 114489,
    'duplicates_removed': 6052,
    'n_blank_original_post_id': 1,
    'ordered_post_ids_sha256': hashlib.sha256(
        '\n'.join(
            master.post_id.astype(str)
        ).encode()
    ).hexdigest(),
    'label_sha256': INPUT_SHA,
    'text_origin':
        'canonical master title + newline + selftext',
    'text_audit': text_audit_summary
}

save_json(
    AUDIT / 'corpus_manifest.json',
    master_manifest
)

print(
    'MASTER PASS:',
    len(master),
    'unique posts; duplicate reduction 6052'
)

print('Text audit:', text_audit_summary)


In [ ]:

# ============================================================
# CELL 03A — MANUAL REVIEW OF TRUNCATED REDDIT POST (BR0325)
# ============================================================
#
# Purpose:
# Examine the additional text of a Reddit post that appears
# to have been truncated in the original manual-review Excel.
#
# Background:
# - Audit ID: BR0325
# - Reddit Post ID: stw7uq
# - Original reviewer decision: Exclude
# - The canonical master corpus contains approximately 4,400
#   additional characters compared with the reviewed text.
#
# Objective:
# Determine whether the previously unseen content contains
# evidence that could affect the original exclusion decision.
#
# Primary inclusion criteria:
# 1. First-person disclosure of a gaming-related problem.
# 2. Current, ongoing, or unresolved problem.
# 3. At least one of:
#    - Impaired control over gaming
#    - Meaningful gaming-related psychological distress
#    - Meaningful negative consequences
#
# Important:
# This cell is diagnostic only. It does NOT change labels,
# datasets, reviewer decisions, or model training inputs.
# ============================================================

from pathlib import Path

# Locate the additional text extracted during Cell 03.
suffix_path = (
    AUDIT / "BR0325_unseen_master_suffix_PRIVATE.txt"
)

assert suffix_path.exists(), (
    "Missing additional text. Run Cell 03 first."
)

# Read the previously unseen portion of the Reddit post.
suffix = suffix_path.read_text(encoding="utf-8")

# Display the content for manual inspection.
print("=" * 70)
print("MANUAL REVIEW — BR0325")
print("Reddit Post ID: stw7uq")
print("Current label: Exclude")
print("Additional characters:", len(suffix))
print("=" * 70)

print("\nPREVIOUSLY UNSEEN POST CONTENT:\n")
print(suffix)

print("\n" + "=" * 70)
print("REVIEW REQUIRED")
print(
    "Check whether this text changes the original "
    "primary inclusion/exclusion decision."
)
print(
    "Do not change the label without documenting "
    "the reviewer decision and its rationale."
)
print("=" * 70)


In [ ]:

# ============================================================
# CELL 04 — FINAL DATASET CONSTRUCTION AND AUTHOR-GROUP AUDIT
# ============================================================
#
# Purpose:
# Rebuild all RQ2 datasets from the final adjudicated labels
# and the canonical, source-verified r/MMORPG corpus.
#
# Important methodological decisions:
# - Retain all manually reviewed posts, including posts whose
#   author field is [deleted].
# - Exclude the 7 Uncertain posts from binary classification.
# - Exclude all 365 screened candidates from community controls.
# - Treat deleted/anonymous authors as unlinked singleton
#   analysis groups, consistent with the previous manuscript.
# - Preserve author-grouped evaluation for identifiable authors.
# - Use 58:116 and 18:36 as the primary 1:2 comparisons.
# - Preserve matched and hard-negative sensitivity analyses.
# - Do not recreate the historical 415-control experiments.
# ============================================================

def reviewed_to_master(rows, role):
    out = master.loc[
        master.post_id.isin(set(rows.post_id))
    ].copy()

    out = out.merge(
        rows[['post_id', 'audit_id', 'final_decision']],
        on='post_id',
        validate='one_to_one'
    )

    out['label'] = int(role == 'Include')
    out['text'] = out.full_text

    return out[
        [
            'audit_id', 'post_id', 'text',
            'analysis_group', 'label',
            'word_len', 'char_len'
        ]
    ].copy()


# ------------------------------------------------------------
# FINAL REVIEWED POSITIVES AND HARD NEGATIVES
# ------------------------------------------------------------

positives = reviewed_to_master(
    positives_reviewed, 'Include'
)

negatives = reviewed_to_master(
    negatives_reviewed, 'Exclude'
)

assert len(positives) == 58
assert len(negatives) == 300

all_screened_ids = set(labels.post_id)

community = master.loc[
    ~master.post_id.isin(all_screened_ids)
].copy()

strict = community.loc[
    ~community.full_text.str.contains(
        STRICT5, na=False
    )
].copy()

strict_linkable = strict.loc[
    strict.post_id.ne('')
].copy()

assert len(community) == 114124
assert len(strict_linkable) >= 116


def controls_view(src):
    out = src[
        [
            'post_id', 'full_text', 'analysis_group',
            'word_len', 'char_len'
        ]
    ].rename(columns={'full_text': 'text'}).copy()

    out['audit_id'] = ''
    out['label'] = 0

    return out[
        [
            'audit_id', 'post_id', 'text',
            'analysis_group', 'label',
            'word_len', 'char_len'
        ]
    ]


# ------------------------------------------------------------
# FIXED COMMUNITY COMPARISON SAMPLES
# ------------------------------------------------------------

# Primary 1:2 comparison.
neg_116 = strict_linkable.sample(
    n=116, replace=False, random_state=42
)

# Strict 1:2 comparison.
neg_36 = neg_116.iloc[:36].copy()

# Unmatched random 1:1 sensitivity.
random_controls = strict_linkable.sample(
    n=58, replace=False, random_state=42
)


# ------------------------------------------------------------
# QUANTILE-BASED LENGTH MATCHING
# ------------------------------------------------------------
#
# Preserve the five-bin character-length matching approach
# used in the previous revision.
# ------------------------------------------------------------

combined_lengths = pd.concat(
    [
        positives[['char_len']],
        strict[['char_len']]
    ],
    ignore_index=True
)

char_bins = pd.qcut(
    combined_lengths.char_len,
    q=5,
    labels=False,
    duplicates='drop'
)

positive_bins = char_bins.iloc[
    :len(positives)
].reset_index(drop=True)

strict_bins = char_bins.iloc[
    len(positives):
].reset_index(drop=True)

strict_binned = strict.copy().reset_index(drop=True)
strict_binned['len_bin'] = strict_bins

match_parts = []

for bin_value in sorted(
    positive_bins.dropna().unique()
):
    n = int((positive_bins == bin_value).sum())

    candidates = strict_binned.loc[
        (strict_binned.len_bin == bin_value)
        & strict_binned.post_id.ne('')
    ]

    assert len(candidates) >= n

    match_parts.append(
        candidates.sample(
            n=n,
            replace=False,
            random_state=42
        )
    )

quantile_controls = pd.concat(
    match_parts, ignore_index=True
)

assert len(quantile_controls) == 58
assert quantile_controls.post_id.is_unique


# ------------------------------------------------------------
# TIGHT 1:1 LENGTH MATCHING
# ------------------------------------------------------------
#
# Reproduce the previous R1 approach:
# - log1p character length and word length
# - standardisation
# - nearest-neighbour candidate controls
# - global greedy assignment
# - no replacement of comparison posts
# ------------------------------------------------------------

from sklearn.neighbors import NearestNeighbors

match_positives = positives.reset_index(drop=True)
match_pool = strict_linkable.reset_index(drop=True)

pos_match_features = np.column_stack([
    np.log1p(
        match_positives.char_len.to_numpy(float)
    ),
    np.log1p(
        match_positives.word_len.to_numpy(float)
    ),
])

ctrl_match_features = np.column_stack([
    np.log1p(
        match_pool.char_len.to_numpy(float)
    ),
    np.log1p(
        match_pool.word_len.to_numpy(float)
    ),
])

match_scaler = StandardScaler()

match_scaler.fit(
    np.vstack([
        pos_match_features,
        ctrl_match_features
    ])
)

pos_scaled = match_scaler.transform(
    pos_match_features
)

ctrl_scaled = match_scaler.transform(
    ctrl_match_features
)

K_NEIGHBORS = min(200, len(match_pool))

nn = NearestNeighbors(
    n_neighbors=K_NEIGHBORS,
    metric='euclidean',
    algorithm='auto'
)

nn.fit(ctrl_scaled)

near_dist, near_idx = nn.kneighbors(
    pos_scaled
)

candidate_pairs = sorted(
    (
        float(near_dist[i, j]),
        i,
        int(near_idx[i, j])
    )
    for i in range(len(match_positives))
    for j in range(K_NEIGHBORS)
)

assigned_positive = set()
assigned_control = set()
matches = []

for distance, pi, ci in candidate_pairs:
    if pi in assigned_positive or ci in assigned_control:
        continue

    assigned_positive.add(pi)
    assigned_control.add(ci)

    matches.append((pi, ci, distance))

    if len(matches) == len(match_positives):
        break

# Fallback when an unmatched positive remains after
# the initial 200-neighbour candidate assignment.
if len(matches) < len(match_positives):

    for pi in range(len(match_positives)):
        if pi in assigned_positive:
            continue

        remaining = np.asarray(
            [
                i for i in range(len(match_pool))
                if i not in assigned_control
            ],
            dtype=int
        )

        diff = ctrl_scaled[remaining] - pos_scaled[pi]

        dist = np.sqrt(
            np.sum(diff ** 2, axis=1)
        )

        best = int(np.argmin(dist))
        ci = int(remaining[best])

        matches.append(
            (pi, ci, float(dist[best]))
        )

        assigned_positive.add(pi)
        assigned_control.add(ci)

assert len(matches) == 58
assert len({pi for pi, ci, dist in matches}) == 58
assert len({ci for pi, ci, dist in matches}) == 58

matches.sort(key=lambda t: t[0])

chosen = [
    ci for pi, ci, dist in matches
]

tight_controls = match_pool.iloc[
    chosen
].copy().reset_index(drop=True)

assert len(tight_controls) == 58
assert tight_controls.post_id.is_unique

pair_audit = []

for pi, ci, dist in matches:
    pr = match_positives.iloc[pi]
    cr = match_pool.iloc[ci]

    pair_audit.append({
        'positive_post_id': pr.post_id,
        'control_post_id': cr.post_id,
        'positive_word_len': int(pr.word_len),
        'control_word_len': int(cr.word_len),
        'positive_char_len': int(pr.char_len),
        'control_char_len': int(cr.char_len),
        'absolute_word_difference': abs(
            int(pr.word_len) - int(cr.word_len)
        ),
        'absolute_char_difference': abs(
            int(pr.char_len) - int(cr.char_len)
        ),
        'standardized_log_length_distance': float(dist)
    })

atomic_csv(
    pd.DataFrame(pair_audit),
    DATA / 'tight_length_matching_pairs_PRIVATE.csv'
)


# ------------------------------------------------------------
# STRICT HIGH-SPECIFICITY SUBSET (18 POSTS)
# ------------------------------------------------------------

strict_ids = set(
    strict_rows.post_id.astype(str)
)

strict_positives = positives.loc[
    positives.post_id.isin(strict_ids)
].copy()

assert len(strict_positives) == 18
assert strict_positives.post_id.nunique() == 18
assert set(strict_positives.post_id) == strict_ids


# ------------------------------------------------------------
# ANONYMOUS / DELETED AUTHOR AUDIT
# ------------------------------------------------------------
#
# All seven [deleted]-author posts remain in the dataset.
#
# The reviewer Excel contains a shared hash of the literal
# '[deleted]' placeholder. This is not a real author identity.
#
# The canonical master instead assigns these posts separate
# analysis groups, one for each Post ID.
#
# This is consistent with the previous manuscript's policy
# for anonymous / non-linkable authors.
# ------------------------------------------------------------

DELETED_REVIEW_HASH = hashlib.sha256(
    b'[deleted]'
).hexdigest()[:16]

review_vs_master_groups = strict_positives[
    ['post_id', 'analysis_group']
].merge(
    strict_rows[
        ['post_id', 'review_author_group']
    ],
    on='post_id',
    validate='one_to_one'
)

assert len(review_vs_master_groups) == 18

is_deleted = (
    review_vs_master_groups.review_author_group
    .eq(DELETED_REVIEW_HASH)
)

assert int(is_deleted.sum()) == 7

deleted_groups = review_vs_master_groups.loc[
    is_deleted
].copy()

expected_anonymous_group = (
    'anonymous__' + deleted_groups.post_id.astype(str)
)

if not deleted_groups.analysis_group.eq(
    expected_anonymous_group
).all():

    atomic_csv(
        review_vs_master_groups,
        AUDIT / 'BLOCKING_strict_author_group_mismatch_PRIVATE.csv'
    )

    raise ValueError(
        'A [deleted] author was not assigned the expected '
        'singleton analysis group. Inspect the saved audit.'
    )


# ------------------------------------------------------------
# KNOWN-AUTHOR GROUP CONSISTENCY
# ------------------------------------------------------------

known_groups = review_vs_master_groups.loc[
    ~is_deleted
].copy()

review_to_master = known_groups.groupby(
    'review_author_group'
).analysis_group.nunique()

master_to_review = known_groups.groupby(
    'analysis_group'
).review_author_group.nunique()

if (
    review_to_master.max() > 1
    or master_to_review.max() > 1
):

    atomic_csv(
        review_vs_master_groups,
        AUDIT / 'BLOCKING_strict_author_group_mismatch_PRIVATE.csv'
    )

    raise ValueError(
        'Author grouping differs between the reviewer sheet '
        'and canonical master. Inspect the mapping.'
    )

EXPECTED_STRICT_ANALYSIS_GROUPS = (
    known_groups.review_author_group.nunique()
    + len(deleted_groups)
)

ACTUAL_STRICT_ANALYSIS_GROUPS = (
    strict_positives.analysis_group.nunique()
)

assert (
    ACTUAL_STRICT_ANALYSIS_GROUPS
    == EXPECTED_STRICT_ANALYSIS_GROUPS
), (
    'Strict analysis-group count differs from the '
    'documented anonymous-author handling.'
)

atomic_csv(
    review_vs_master_groups,
    AUDIT / 'strict18_author_group_provenance_PRIVATE.csv'
)

print(
    'STRICT AUTHOR GROUP PASS:',
    ACTUAL_STRICT_ANALYSIS_GROUPS,
    'analysis groups;',
    len(deleted_groups),
    'deleted-author singleton posts;',
    known_groups.review_author_group.nunique(),
    'identifiable reviewer group tokens.'
)


# ------------------------------------------------------------
# UPDATE REVIEWER / LABEL PROVENANCE MANIFEST
# ------------------------------------------------------------

label_manifest.update({
    'strict_v2_analysis_groups_canonical':
        int(ACTUAL_STRICT_ANALYSIS_GROUPS),

    'strict_v2_deleted_author_singletons':
        int(len(deleted_groups)),

    'strict_v2_group_interpretation': (
        'All [deleted]-author posts were retained. '
        'The reviewer Excel groups their placeholder under '
        'one hash, while canonical cross-validation treats '
        'each such post as an unlinked singleton. '
        'Analysis groups are not a count of identifiable '
        'human authors.'
    )
})

save_json(
    AUDIT / 'label_manifest.json',
    label_manifest
)


# ------------------------------------------------------------
# STRICT LENGTH-MATCHED SENSITIVITY
# ------------------------------------------------------------
#
# Inherit comparison controls from the primary 58-pair
# tight matching. Do not select based on model performance.
# ------------------------------------------------------------

strict_pairs = pd.DataFrame(pair_audit).loc[
    lambda z: z.positive_post_id.isin(strict_ids)
].copy()

assert len(strict_pairs) == 18
assert strict_pairs.control_post_id.is_unique

strict_tight_controls = (
    strict_linkable.set_index(
        'post_id', drop=False
    ).loc[
        strict_pairs.control_post_id.astype(str).tolist()
    ].reset_index(drop=True)
)

assert len(strict_tight_controls) == 18
assert strict_tight_controls.post_id.is_unique

assert not (
    set(strict_tight_controls.post_id)
    & set(labels.post_id)
)

atomic_csv(
    strict_pairs,
    DATA / 'strict18_v2_tight_matching_pairs_PRIVATE.csv'
)


# ------------------------------------------------------------
# REVIEWED HARD-NEGATIVE SAMPLES
# ------------------------------------------------------------

hard_116 = negatives.sample(
    n=116,
    replace=False,
    random_state=42
).copy()

hard_36 = negatives.sample(
    n=36,
    replace=False,
    random_state=42
).copy()

assert set(neg_36.post_id).issubset(
    set(neg_116.post_id)
)

assert set(hard_36.post_id).issubset(
    set(hard_116.post_id)
)


# ------------------------------------------------------------
# GENERIC DATASET CONSTRUCTION HELPER
# ------------------------------------------------------------

def combine_task(pos, neg, name):

    frame = pd.concat(
        [pos, neg],
        ignore_index=True
    )

    assert frame.post_id.is_unique, name
    assert frame.post_id.ne('').all(), name

    assert not (
        set(frame.post_id)
        & set(uncertain_reviewed.post_id)
    ), name

    assert set(frame.label.unique()) == {0, 1}, name
    assert len(frame) == len(pos) + len(neg)
    assert int(frame.label.sum()) == len(pos)

    return frame


# ------------------------------------------------------------
# FINAL RQ2 DATASET REGISTRY
# ------------------------------------------------------------

DATASETS = {
    'rq2_primary_58v116': combine_task(
        positives,
        controls_view(neg_116),
        'primary_1to2'
    ),

    'rq2_primary_58v116_hardneg': combine_task(
        positives,
        hard_116,
        'primary_hard_1to2'
    ),

    'rq2_candidate_58v300': combine_task(
        positives,
        negatives,
        'primary_all_hard'
    ),

    'rq2_random_58v58': combine_task(
        positives,
        controls_view(random_controls),
        'primary_random_1to1'
    ),

    'rq2_quantile_58v58': combine_task(
        positives,
        controls_view(quantile_controls),
        'primary_quantile_1to1'
    ),

    'rq2_tight_58v58': combine_task(
        positives,
        controls_view(tight_controls),
        'primary_tight_1to1'
    ),

    'rq2_strict18v36': combine_task(
        strict_positives,
        controls_view(neg_36),
        'strict_1to2'
    ),

    'rq2_strict18v36_hardneg': combine_task(
        strict_positives,
        hard_36,
        'strict_hard_1to2'
    ),

    'rq2_strict18v300': combine_task(
        strict_positives,
        negatives,
        'strict_all_hard'
    ),

    'rq2_strict18v18': combine_task(
        strict_positives,
        controls_view(strict_tight_controls),
        'strict_tight_1to1'
    )
}

STRICT_DATASETS = {
    name: frame
    for name, frame in DATASETS.items()
    if name.startswith('rq2_strict')
}

EXPECTED_TASK_COUNTS = {
    'rq2_primary_58v116': (58, 116),
    'rq2_primary_58v116_hardneg': (58, 116),
    'rq2_candidate_58v300': (58, 300),
    'rq2_random_58v58': (58, 58),
    'rq2_quantile_58v58': (58, 58),
    'rq2_tight_58v58': (58, 58),
    'rq2_strict18v36': (18, 36),
    'rq2_strict18v36_hardneg': (18, 36),
    'rq2_strict18v300': (18, 300),
    'rq2_strict18v18': (18, 18)
}

assert set(DATASETS) == set(EXPECTED_TASK_COUNTS)

for name, frame in DATASETS.items():

    n_pos, n_neg = EXPECTED_TASK_COUNTS[name]

    assert (
        int(frame.label.sum()),
        int((frame.label == 0).sum())
    ) == (n_pos, n_neg), name

    atomic_csv(
        frame,
        DATA / (name + '_PRIVATE.csv')
    )

    print(
        'R2 DATASET:',
        name,
        '| positives:', n_pos,
        '| comparisons:', n_neg,
        '| analysis groups:',
        frame.analysis_group.nunique()
    )


# ------------------------------------------------------------
# DOCUMENT THE REVISED STRICT SUBSET
# ------------------------------------------------------------

save_json(
    DATA / 'high_specificity_v2_status.json',
    {
        'status': 'POSTHOC_PROPOSED_STRICT_18_V2',
        'n_primary': 58,
        'n_strict': 18,
        'historical_removed': list(
            STRICT_V2_SWAPS.keys()
        ),
        'proposed_added': list(
            STRICT_V2_SWAPS.values()
        ),
        'human_strict_reconfirmation': 'PENDING',
        'strict_rule': 'F AND T AND C AND (D OR H)',
        'strict_post_ids': sorted(strict_ids),
        'label_sha256': INPUT_SHA,
        'analysis_groups': int(
            ACTUAL_STRICT_ANALYSIS_GROUPS
        ),
        'deleted_author_singletons': int(
            len(deleted_groups)
        ),
        'max_posts_per_analysis_group': int(
            strict_positives.analysis_group
            .value_counts().max()
        ),
        'dataset_sha256': {
            k: sha256_file(
                DATA / (k + '_PRIVATE.csv')
            )
            for k in STRICT_DATASETS
        }
    }
)


# ------------------------------------------------------------
# FINAL DATASET MANIFEST
# ------------------------------------------------------------

pool_manifest = {
    'label_sha256': INPUT_SHA,
    'master_sha256': MASTER_SHA,
    'final_positive': 58,
    'final_reviewed_negative': 300,
    'excluded_uncertain': 7,
    'all_reviewed_blocked_from_community': 365,
    'community_pool_n': len(community),
    'strict_pool_n': len(strict),
    'primary_ratio':
        '58 positive:116 strict community comparisons',
    'strict_ratio':
        '18 positive:36 strict community comparisons',
    'hard_negative_ratio_checks': [
        '58 positive:116 reviewed Exclude',
        '18 positive:36 reviewed Exclude'
    ],
    'all_reviewed_hard_negative_challenges': [
        '58 positive:300 Exclude',
        '18 positive:300 Exclude'
    ],
    'community_sample_seed': 42,
    'reviewed_hard_negative_seed': 42,
    'random_58_seed': 42,
    'quantile_matching':
        '5 joint character-length bins, seed 42, '
        'without replacement',
    'tight_matching':
        'standardized log1p(char+word length), '
        '200 nearest controls, global greedy '
        'unique-pair assignment',
    'dataset_files': {
        k: sha256_file(
            DATA / (k + '_PRIVATE.csv')
        )
        for k in DATASETS
    }
}

save_json(
    DATA / 'dataset_manifest.json',
    pool_manifest
)

assert len(DATASETS) == 10
assert len(STRICT_DATASETS) == 4

assert '415' not in ' '.join(DATASETS), (
    'Old 415-control scenario was unexpectedly included.'
)

print(
    '\nFINAL v6 DATASET AUDIT PASS:',
    '10 RQ2 scenarios, 4 strict,',
    'no old 415 controls.'
)


In [ ]:
# 05 — Length and authorship diagnostics; never call a sample matched without checking it.
rows=[]
for task,frame in DATASETS.items():
    a=frame.loc[frame.label.eq(1)]
    b=frame.loc[frame.label.eq(0)]
    for feature in ('word_len','char_len'):
        aa=a[feature].to_numpy(float);bb=b[feature].to_numpy(float)
        sd=np.sqrt((aa.var(ddof=1)+bb.var(ddof=1))/2)
        rows.append({'scenario':task,'feature':feature,'n_positive':len(a),'n_control':len(b),
            'positive_median':float(np.median(aa)),'control_median':float(np.median(bb)),
            'positive_Q1':float(np.quantile(aa,.25)),'positive_Q3':float(np.quantile(aa,.75)),
            'control_Q1':float(np.quantile(bb,.25)),'control_Q3':float(np.quantile(bb,.75)),
            'length_cohen_d':float((aa.mean()-bb.mean())/sd) if sd>0 else 0,
            'unique_author_groups':frame.analysis_group.nunique()})
length_audit=pd.DataFrame(rows)
atomic_csv(length_audit,DATA/'length_and_author_audit.csv')
print(length_audit[['scenario','feature','positive_median','control_median','length_cohen_d']].round(4).to_string(index=False))

In [ ]:

# ============================================================
# CELL 06 — GROUPED CROSS-VALIDATION AND CLASSICAL BASELINES
# ============================================================
#
# Purpose:
# 1. Construct reproducible author-grouped cross-validation
#    folds for all ten final RQ2 datasets.
# 2. Prevent author overlap between training and test sets.
# 3. Ensure that every test fold contains both classes.
# 4. Evaluate lexical, TF-IDF, and length-only baselines.
# 5. Save out-of-fold predictions and reproducibility records.
#
# Methodological safeguards:
# - Five-fold StratifiedGroupKFold.
# - Identifiable authors cannot cross train/test boundaries.
# - Deleted/anonymous authors retain singleton groups.
# - The first valid seed is selected from a fixed sequence.
# - Split selection depends only on labels and author groups.
# - No AUC, predictions, or model performance are used
#   to choose the cross-validation seed.
# - TF-IDF vocabulary and scaling are fitted within folds.
# - The same frozen splits are used by subsequent models.
#
# Primary scenarios:
#   58 positive vs 116 community comparisons
#   18 strict positive vs 36 community comparisons
#
# Additional scenarios:
#   Reviewed hard negatives and length-matched controls.
#
# No historical 415-control experiments are included.
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import numpy as np
import pandas as pd
import hashlib
import re


# ------------------------------------------------------------
# 1. REPRODUCIBLE AUTHOR-GROUPED CV
# ------------------------------------------------------------

CV_SELECTED_SEEDS = {}
CV_FOLD_DIAGNOSTICS = []


def fixed_cv(
    frame,
    task_name,
    min_test_per_class=2,
    max_seeds=2000
):
    """
    Generate five author-grouped folds with both classes
    represented in every train and test partition.

    Search candidate seeds in ascending order, starting
    from CV_SEED (42). Retain the first valid partition.

    Split selection uses only labels and author groups,
    never model predictions or performance.
    """

    y = frame["label"].to_numpy(dtype=int)

    groups = (
        frame["analysis_group"]
        .astype(str)
        .to_numpy()
    )

    assert set(np.unique(y)) == {0, 1}, (
        f"{task_name}: expected binary labels."
    )

    class_counts = np.bincount(y, minlength=2)

    if class_counts.min() < N_FOLDS * min_test_per_class:
        raise ValueError(
            f"{task_name}: insufficient cases for "
            f"{N_FOLDS} class-complete folds."
        )

    chosen_splits = None
    chosen_seed = None

    # Deterministic search, independent of model performance.
    for offset in range(max_seeds):

        candidate_seed = int(CV_SEED) + offset

        cv = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=candidate_seed
        )

        candidate_splits = list(
            cv.split(
                np.zeros(len(frame)),
                y,
                groups
            )
        )

        valid = True

        for train_idx, test_idx in candidate_splits:

            train_groups = set(groups[train_idx])
            test_groups = set(groups[test_idx])

            # Author groups must not cross partitions.
            if not train_groups.isdisjoint(test_groups):
                raise AssertionError(
                    f"Author leakage in {task_name}."
                )

            train_counts = np.bincount(
                y[train_idx],
                minlength=2
            )

            test_counts = np.bincount(
                y[test_idx],
                minlength=2
            )

            # Both classes in training and at least
            # two examples per class in testing.
            if (
                train_counts.min() == 0
                or test_counts.min() < min_test_per_class
            ):
                valid = False
                break

        if valid:
            chosen_splits = candidate_splits
            chosen_seed = candidate_seed
            break

    if chosen_splits is None:
        raise ValueError(
            f"{task_name}: no valid five-fold grouped split "
            f"found among {max_seeds} deterministic seeds. "
            "Inspect the author-group distribution."
        )

    # Record each held-out fold.
    fold_assignments = np.zeros(
        len(frame),
        dtype=int
    )

    all_test_indices = []

    for fold_no, (train_idx, test_idx) in enumerate(
        chosen_splits,
        start=1
    ):

        train_counts = np.bincount(
            y[train_idx],
            minlength=2
        )

        test_counts = np.bincount(
            y[test_idx],
            minlength=2
        )

        assert set(groups[train_idx]).isdisjoint(
            set(groups[test_idx])
        )

        assert train_counts.min() > 0
        assert test_counts.min() >= min_test_per_class

        # No post should appear in multiple test folds.
        assert np.all(
            fold_assignments[test_idx] == 0
        )

        fold_assignments[test_idx] = fold_no

        all_test_indices.extend(
            test_idx.tolist()
        )

        CV_FOLD_DIAGNOSTICS.append({
            "scenario": task_name,
            "selected_split_seed": chosen_seed,
            "fold": fold_no,
            "train_negative": int(train_counts[0]),
            "train_positive": int(train_counts[1]),
            "test_negative": int(test_counts[0]),
            "test_positive": int(test_counts[1]),
            "test_groups": len(
                set(groups[test_idx])
            )
        })

    # Every post must appear in exactly one test fold.
    assert np.array_equal(
        np.sort(all_test_indices),
        np.arange(len(frame))
    )

    assert (fold_assignments > 0).all()

    CV_SELECTED_SEEDS[task_name] = chosen_seed

    return chosen_splits, fold_assignments


# ------------------------------------------------------------
# 2. FEATURE EXTRACTION
# ------------------------------------------------------------

def length_features(texts):
    """
    Return word count and character count.

    This baseline measures how much classification
    performance can be explained by document length.
    """

    return np.asarray(
        [
            [
                len(str(text).split()),
                len(str(text))
            ]
            for text in texts
        ],
        dtype=float
    )


def lexical_features(texts):
    """
    Fixed lexical indicators relevant to addiction-
    related screening and gaming-related disclosures.

    These features are deliberately simple to help
    quantify the contribution of explicit vocabulary.
    """

    terms = [
        "addict",
        "addicted",
        "addiction",
        "addictive",
        "hooked",
        "quit",
        "stop",
        "relapse"
    ]

    return np.asarray(
        [
            [
                int(
                    bool(
                        re.search(
                            r"\b" + re.escape(term) + r"\b",
                            str(text),
                            re.I
                        )
                    )
                )
                for term in terms
            ]
            for text in texts
        ],
        dtype=float
    )


# ------------------------------------------------------------
# 3. CLASSICAL BASELINE MODEL FACTORY
# ------------------------------------------------------------

def classical_model(model_name):
    """
    Create a fresh model for each cross-validation fold.

    TF-IDF fitting and feature scaling are performed
    only on training data within the sklearn Pipeline.
    """

    if model_name in (
        "tfidf_raw",
        "tfidf_masked"
    ):

        return Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=5000,
                    sublinear_tf=True,
                    stop_words="english"
                )
            ),
            (
                "clf",
                LogisticRegression(
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=3000,
                    random_state=42
                )
            )
        ])

    # Length-only and lexical-indicator models.
    return Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                solver="liblinear",
                max_iter=3000,
                random_state=42
            )
        )
    ])


# ------------------------------------------------------------
# 4. EVALUATION METRICS
# ------------------------------------------------------------

def evaluate_predictions(y, score):
    """
    Evaluate out-of-fold positive-class probabilities.

    Metrics include ROC-AUC, average precision,
    balanced accuracy, precision, sensitivity,
    specificity, F1, and confusion-matrix counts.

    Binary decisions use the fixed threshold 0.5.
    """

    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)

    assert len(y) == len(score)
    assert np.isfinite(score).all()

    predicted = (
        score >= 0.5
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        predicted,
        labels=[0, 1]
    ).ravel()

    return {
        "ROC_AUC": roc_auc_score(y, score),
        "AP": average_precision_score(y, score),
        "BAcc": balanced_accuracy_score(
            y, predicted
        ),
        "precision": precision_score(
            y, predicted, zero_division=0
        ),
        "recall_sensitivity": recall_score(
            y, predicted, zero_division=0
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "F1": f1_score(
            y, predicted, zero_division=0
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 5. FREEZE CROSS-VALIDATION SPLITS
# ------------------------------------------------------------

ALL_SPLITS = {}
ALL_FOLDS = {}

assert len(DATASETS) == 10

for task_name, frame in DATASETS.items():

    splits, fold_assignments = fixed_cv(
        frame,
        task_name
    )

    ALL_SPLITS[task_name] = splits

    ALL_FOLDS[task_name] = fold_assignments

    test_positive_counts = [
        int(
            frame["label"].iloc[test_idx].sum()
        )
        for _, test_idx in splits
    ]

    print(
        "CV PASS:",
        task_name,
        "| selected seed:",
        CV_SELECTED_SEEDS[task_name],
        "| test positives:",
        test_positive_counts
    )


# ------------------------------------------------------------
# 6. SAVE CROSS-VALIDATION PROVENANCE
# ------------------------------------------------------------

atomic_csv(
    pd.DataFrame(CV_FOLD_DIAGNOSTICS),
    AUDIT / "R2_grouped_cv_fold_diagnostics_PRIVATE.csv"
)

save_json(
    AUDIT / "R2_grouped_cv_seeds.json",
    {
        "method": (
            "StratifiedGroupKFold, "
            "first valid deterministic seed"
        ),
        "candidate_seed_start": CV_SEED,
        "n_folds": N_FOLDS,
        "minimum_each_test_class": 2,
        "performance_based_split_selection": False,
        "selected_seed_by_scenario": CV_SELECTED_SEEDS,
        "fold_assignment_sha256": {
            task_name: hashlib.sha256(
                fold_array.tobytes()
            ).hexdigest()
            for task_name, fold_array in ALL_FOLDS.items()
        }
    }
)

print(
    "\nALL GROUPED CV SPLITS PASS:",
    len(ALL_SPLITS),
    "tasks;",
    N_FOLDS,
    "folds each; no author leakage."
)


# ------------------------------------------------------------
# 7. CLASSICAL AND LEXICAL BASELINES
# ------------------------------------------------------------

BASE_MODELS = [
    "tfidf_raw",
    "tfidf_masked",
    "length_only",
    "lexical_features",
    "strict5_fixed_rule"
]

base_metrics = []
base_predictions = []

for task_name, frame in DATASETS.items():

    texts = (
        frame["text"]
        .astype(str)
        .to_numpy(dtype=object)
    )

    y = frame["label"].to_numpy(
        dtype=int
    )

    # Raw and keyword-masked representations.
    feature_sets = {
        "tfidf_raw": texts,

        "tfidf_masked": np.asarray(
            [
                mask_text(text)
                for text in texts
            ],
            dtype=object
        ),

        "length_only": length_features(texts),

        "lexical_features": lexical_features(texts)
    }

    for model_name in BASE_MODELS:

        scores = np.full(
            len(y),
            np.nan,
            dtype=float
        )

        if model_name == "strict5_fixed_rule":

            # Fixed keyword-presence rule.
            # No fitting or cross-validation training.
            scores = np.asarray(
                [
                    float(
                        bool(STRICT5.search(text))
                    )
                    for text in texts
                ],
                dtype=float
            )

        else:

            # Fit a fresh pipeline within every training fold.
            for fold_no, (
                train_idx, test_idx
            ) in enumerate(
                ALL_SPLITS[task_name],
                start=1
            ):

                fitted_model = classical_model(
                    model_name
                )

                fitted_model.fit(
                    feature_sets[model_name][train_idx],
                    y[train_idx]
                )

                scores[test_idx] = (
                    fitted_model.predict_proba(
                        feature_sets[model_name][test_idx]
                    )[:, 1]
                )

        assert np.isfinite(scores).all()

        metrics = evaluate_predictions(
            y,
            scores
        )

        base_metrics.append({
            "scenario": task_name,
            "model": model_name,
            "n": len(y),
            "positives": int(y.sum()),
            **metrics
        })

        base_predictions.append(
            pd.DataFrame({
                "scenario": task_name,
                "model": model_name,
                "post_id": frame["post_id"],
                "analysis_group": frame["analysis_group"],
                "label": y,
                "cv_fold": ALL_FOLDS[task_name],
                "score": scores
            })
        )

    print(
        "BASELINES COMPLETE:",
        task_name
    )


# ------------------------------------------------------------
# 8. SAVE BASELINE RESULTS
# ------------------------------------------------------------

BASE_METRICS = pd.DataFrame(
    base_metrics
)

BASE_OOF = pd.concat(
    base_predictions,
    ignore_index=True
)

atomic_csv(
    BASE_METRICS,
    BASE / "R2_final_lexical_metrics.csv"
)

atomic_csv(
    BASE_OOF,
    BASE / "R2_final_lexical_OOF_PRIVATE.csv"
)

print("\nALL CLASSICAL BASELINE RESULTS:")

print(
    BASE_METRICS[
        [
            "scenario",
            "model",
            "ROC_AUC",
            "AP",
            "BAcc"
        ]
    ].round(4).to_string(index=False)
)


# ------------------------------------------------------------
# 9. REVIEWER-RELEVANT COMPARISONS
# ------------------------------------------------------------

# Explicitly compare the fixed keyword rule, TF-IDF,
# length-only, and other baselines on the SAME datasets.
#
# Historical AUC values of 0.619 and 0.658 belong to
# the previous analysis and must not be carried forward
# as results of the current 58/18 experiments.

reviewer_scenarios = [
    "rq2_primary_58v116",
    "rq2_primary_58v116_hardneg",
    "rq2_candidate_58v300",
    "rq2_strict18v36",
    "rq2_strict18v36_hardneg"
]

primary_show = BASE_METRICS.loc[
    BASE_METRICS["scenario"].isin(
        reviewer_scenarios
    )
].copy()

atomic_csv(
    primary_show,
    BASE / "R2_1to2_and_hardneg_baseline_summary.csv"
)

print(
    "\nREVIEWER-RELEVANT BASELINES:"
)

print(
    primary_show[
        [
            "scenario",
            "model",
            "ROC_AUC",
            "AP",
            "BAcc"
        ]
    ].round(4).to_string(index=False)
)

print(
    "\nCELL 06 COMPLETE — "
    "10 datasets, grouped CV, classical baselines."
)


In [ ]:

# ============================================================
# CELL 06A — FINAL GROUPED CROSS-VALIDATION AUDIT
# ============================================================
#
# Purpose:
# Verify all frozen cross-validation splits generated by
# Cell 06 using the selected, reproducible split seeds.
#
# Requirements:
# - Five folds per dataset
# - No author-group leakage
# - At least two examples of each class in every test fold
# - Every post appears in exactly one test fold
# - No changes to labels, datasets, or model parameters
# ============================================================

import numpy as np

assert len(ALL_SPLITS) == len(DATASETS) == 10

for task_name, df in DATASETS.items():

    y = df["label"].to_numpy(dtype=int)
    groups = df["analysis_group"].astype(str).to_numpy()

    splits = ALL_SPLITS[task_name]
    assert len(splits) == 5

    all_test_indices = []

    for fold_no, (train_idx, test_idx) in enumerate(
        splits, start=1
    ):
        # Check author-group independence.
        assert set(groups[train_idx]).isdisjoint(
            set(groups[test_idx])
        ), f"Author leakage: {task_name}, fold {fold_no}"

        # Check that both classes are represented.
        train_counts = np.bincount(
            y[train_idx], minlength=2
        )
        test_counts = np.bincount(
            y[test_idx], minlength=2
        )

        assert train_counts.min() > 0
        assert test_counts.min() >= 2

        all_test_indices.extend(test_idx.tolist())

    # Every post must be tested exactly once.
    assert np.array_equal(
        np.sort(all_test_indices),
        np.arange(len(df))
    )

    print(
        f"PASS: {task_name} | "
        f"5 folds | "
        f"seed={CV_SELECTED_SEEDS[task_name]}"
    )

print("\nFINAL GROUPED CV AUDIT: PASS")


In [ ]:
# 07 — Author-cluster bootstrap for selected fixed baseline metrics.
# Resample pseudonymised authors, never individual posts; labels remain fixed.
def clustered_bootstrap(y,p,groups,n_boot=N_BOOT,seed=2026):
    y=np.asarray(y,int);p=np.asarray(p,float);groups=np.asarray(groups,str)
    unique,inv=np.unique(groups,return_inverse=True)
    idx_by_group=[np.flatnonzero(inv==i) for i in range(len(unique))]
    rng=np.random.default_rng(seed);aucs=[];aps=[]
    for _ in range(n_boot):
        chosen=rng.integers(len(unique),size=len(unique))
        idx=np.concatenate([idx_by_group[i] for i in chosen])
        if np.unique(y[idx]).size<2:continue
        aucs.append(roc_auc_score(y[idx],p[idx]))
        aps.append(average_precision_score(y[idx],p[idx]))
    assert len(aucs)>=int(.95*n_boot)
    return {'AUC_CI_low':np.quantile(aucs,.025),'AUC_CI_high':np.quantile(aucs,.975),
        'AP_CI_low':np.quantile(aps,.025),'AP_CI_high':np.quantile(aps,.975),'valid_boots':len(aucs)}

ci_rows=[]
for task in ('rq2_primary_58v116','rq2_primary_58v116_hardneg',
             'rq2_candidate_58v300','rq2_strict18v36',
             'rq2_strict18v36_hardneg','rq2_strict18v300'):
    for model_name in BASE_MODELS:
        x=BASE_OOF.loc[(BASE_OOF.scenario==task)&(BASE_OOF.model==model_name)]
        ci=clustered_bootstrap(x.label,x.score,x.analysis_group)
        ci_rows.append({'scenario':task,'model':model_name,**ci})
CI_BASE=pd.DataFrame(ci_rows)
atomic_csv(CI_BASE,BASE/'R2_final_baseline_clustered_CIs_5000.csv')
print('BASELINE GROUPED CIs PASS')

## RQ1 — Pretrained full-corpus MentalBERT representation

**Only this layer can be reused from the provisional run.** It contains 114,489 embeddings computed *before* any supervised training, with fixed model revision, 512-token truncation, 768 dimensions and mean pooling. This section validates the source and row order before reuse. If validation fails, compute a **fresh** cache; never point an old supervised result at the new labels.

In [ ]:
# 08 — Validate/reuse PRETRAINED embeddings, or compute missing chunks.
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling
from numpy.lib.format import open_memmap
from tqdm.auto import tqdm

EXPECTED_SIG={
    'master_csv_sha256':MASTER_SHA,
    'ordered_post_ids_sha256':master_manifest['ordered_post_ids_sha256'],
    'n':114489,'model':MODEL_NAME,'model_revision':MODEL_REVISION,
    'max_tokens':512,'embedding_dimension':768,
    'embedding_method':'SentenceTransformer mean pooling',
    'normalize_embeddings':False,'chunk_size':1024
}

def cache_is_compatible(path):
    sigfile=path/'cache_signature.json'
    if not sigfile.exists():return False
    try:
        saved=json.loads(sigfile.read_text(encoding='utf-8'))
        return all(saved.get(k)==v for k,v in EXPECTED_SIG.items())
    except Exception:return False

if cache_is_compatible(OLD_EMBED_CACHE):
    EMBED_CACHE=OLD_EMBED_CACHE
    print('Using source-verified historical pretrained embeddings only.')
else:
    EMBED_CACHE=NEW_EMBED_CACHE
    EMBED_CACHE.mkdir(parents=True,exist_ok=True)
    sig=EMBED_CACHE/'cache_signature.json'
    if sig.exists():
        assert cache_is_compatible(EMBED_CACHE),'Local cache signature mismatch. Stop.'
    else:
        save_json(sig,EXPECTED_SIG)

ranges=[(start,min(start+1024,len(master))) for start in range(0,len(master),1024)]
chunk_paths=[EMBED_CACHE/f'chunk_{start:06d}_{end:06d}.npy' for start,end in ranges]
missing=[(start,end,path) for (start,end),path in zip(ranges,chunk_paths) if not path.exists()]
print('Embedding cache:',EMBED_CACHE,'Missing chunks:',len(missing),'of',len(ranges))

if missing:
    assert torch.cuda.is_available(),'A GPU is required to compute missing embeddings.'
    encoder=SentenceTransformer(MODEL_NAME,revision=MODEL_REVISION,device='cuda')
    encoder.max_seq_length=512
    pooling=[m for m in encoder._modules.values() if isinstance(m,Pooling)]
    assert len(pooling)==1
    modes=getattr(pooling[0],'pooling_mode',None)
    if modes is None:
        assert getattr(pooling[0],'pooling_mode_mean_tokens',False)
        assert not any(getattr(pooling[0],k,False) for k in
            ('pooling_mode_cls_token','pooling_mode_max_tokens','pooling_mode_mean_sqrt_len_tokens',
             'pooling_mode_weightedmean_tokens','pooling_mode_lasttoken'))
    else:
        assert (modes==['mean']) if not isinstance(modes,str) else (modes=='mean'),f'Unexpected pooling {modes}'
    dim=(encoder.get_embedding_dimension() if hasattr(encoder,'get_embedding_dimension')
         else encoder.get_sentence_embedding_dimension())
    assert dim==768
    print('Mean pooling PASS, dimension:',dim)
    for start,end,path in tqdm(missing,desc='MentalBERT chunks'):
        values=encoder.encode(master.full_text.iloc[start:end].astype(str).tolist(),batch_size=16,
            show_progress_bar=False,convert_to_numpy=True,normalize_embeddings=False)
        values=np.asarray(values,dtype=np.float32)
        assert values.shape==(end-start,768) and np.isfinite(values).all()
        partial=path.with_name(path.stem+'.partial.npy')
        np.save(partial,values);os.replace(partial,path)
    del encoder;gc.collect();torch.cuda.empty_cache()

for (start,end),path in zip(ranges,chunk_paths):
    part=np.load(path,mmap_mode='r')
    assert part.shape==(end-start,768) and part.dtype==np.float32
    del part

matrix_path=EMBED_CACHE/'all_114489_float32.npy'
if not matrix_path.exists():
    tmp=EMBED_CACHE/'all_114489_float32.partial.npy'
    mem=open_memmap(tmp,mode='w+',dtype='float32',shape=(114489,768))
    for (start,end),path in tqdm(zip(ranges,chunk_paths),total=len(ranges),desc='Build matrix'):
        mem[start:end]=np.load(path,mmap_mode='r')
    mem.flush();del mem;os.replace(tmp,matrix_path)
embeddings=np.load(matrix_path,mmap_mode='r')
assert embeddings.shape==(114489,768) and embeddings.dtype==np.float32
lookup=dict(zip(master.post_id.astype(str),range(len(master))))
assert len(lookup)==114489
save_json(RQ1/'embedding_cache_used.json',{'cache':str(EMBED_CACHE),**EXPECTED_SIG,
    'matrix_file_sha256':sha256_file(matrix_path)})
print('EMBEDDINGS PASS',embeddings.shape)

In [ ]:
# 09 — RQ1 semantic PCA, independent Welch tests, FDR, and one-author sensitivity.
# PCA is separately fitted in each condition. Do NOT compare PC numbers/signs across conditions.
from scipy.spatial.distance import cosine

def rq1_pair(pos,controls):
    p=pos[['post_id','analysis_group']].copy();p['label']=1
    q=controls[['post_id','analysis_group']].copy();q['label']=0
    view=pd.concat([p,q],ignore_index=True)
    assert view.post_id.is_unique and int(view.label.sum()) in (18,58)
    return view

RQ1_CONDITIONS={
    'strict_community':rq1_pair(positives,strict),
    'all_unreviewed_community':rq1_pair(positives,community),
    'five_quantile_char_length':rq1_pair(positives,quantile_controls),
    'random_equal58':rq1_pair(positives,random_controls),
    'tight_one_to_one_length':rq1_pair(positives,tight_controls),
    'primary58_vs_116_1to2':rq1_pair(positives,neg_116),
    'strict18_vs_community':rq1_pair(strict_positives,strict),
    'strict18_vs_36_1to2':rq1_pair(strict_positives,neg_36),
    'strict18_vs_18_tight_matched':rq1_pair(strict_positives,strict_tight_controls)
}

def one_post_per_author(view):
    p=view.loc[view.label.eq(1)].drop_duplicates('analysis_group',keep='first')
    q=view.loc[view.label.eq(0)&~view.analysis_group.isin(set(p.analysis_group))]
    q=q.drop_duplicates('analysis_group',keep='first')
    return pd.concat([p,q],ignore_index=True)

def rq1_semantic(view,condition,variant):
    idx=np.asarray([lookup[str(pid)] for pid in view.post_id],int)
    X=np.asarray(embeddings[idx],dtype=np.float32)
    y=view.label.to_numpy(int)
    assert min(np.bincount(y))>=2
    pcs_n=min(50,len(view)-2,768)
    pca=PCA(n_components=pcs_n,svd_solver='randomized',random_state=42)
    pcs=pca.fit_transform(X)
    pvalues=[];ds=[]
    for j in range(pcs_n):
        a=pcs[y==1,j];b=pcs[y==0,j]
        _,pv=st.ttest_ind(a,b,equal_var=False)
        pvalues.append(float(pv) if np.isfinite(pv) else 1.0)
        sd=math.sqrt((a.var(ddof=1)+b.var(ddof=1))/2)
        ds.append(float((a.mean()-b.mean())/(sd+1e-12)))
    rejected,q,_,_=multipletests(pvalues,alpha=.05,method='fdr_bh')
    details=pd.DataFrame({'condition':condition,'variant':variant,'PC':np.arange(1,pcs_n+1),
        'p_welch':pvalues,'q_FDR_BH':q,'fdr_significant':rejected,
        'cohen_d':ds,'explained_variance_ratio':pca.explained_variance_ratio_})
    summary={'condition':condition,'variant':variant,'n_positive':int(y.sum()),
        'n_controls':int((y==0).sum()),'author_groups':view.analysis_group.nunique(),
        'n_PCs_FDR_significant':int(rejected.sum()),
        'centroid_cosine_distance':float(cosine(X[y==1].mean(axis=0),X[y==0].mean(axis=0)))}
    del X,pcs,pca;gc.collect()
    return details,summary

rq1_details=[];rq1_summary=[]
for condition,view in RQ1_CONDITIONS.items():
    for variant,v in [('all_posts',view),('one_post_per_author',one_post_per_author(view))]:
        detail,summary=rq1_semantic(v,condition,variant)
        rq1_details.append(detail);rq1_summary.append(summary)
        print(condition,variant,'positive',summary['n_positive'],
            'controls',summary['n_controls'],'FDR PCs',summary['n_PCs_FDR_significant'])
RQ1_DETAILS=pd.concat(rq1_details,ignore_index=True)
RQ1_SUMMARY=pd.DataFrame(rq1_summary)
atomic_csv(RQ1_DETAILS,RQ1/'RQ1_PCA50_Welch_FDR_PRIVATE.csv')
atomic_csv(RQ1_SUMMARY,RQ1/'RQ1_semantic_summary.csv')
print('RQ1 PASS')

In [ ]:
# 10 — R1-preserved repeated random equal-size sensitivity (100 draws).
# These 100 draws are repeated comparison samples, not independent cohorts.
if RUN_RQ1_REPEATED_RANDOM:
    repeat_rows=[]
    for repeat in range(100):
        comparison=strict_linkable.sample(n=58,replace=False,random_state=CV_SEED+repeat)
        view=rq1_pair(positives,comparison)
        _,result=rq1_semantic(view,'repeated_random_58v58','all_posts')
        repeat_rows.append({'repeat':repeat+1,'random_seed':CV_SEED+repeat,**result})
    RQ1_RANDOM_REPEATS=pd.DataFrame(repeat_rows)
    atomic_csv(RQ1_RANDOM_REPEATS,RQ1/'RQ1_random_58v58_100_repetitions.csv')
    # Extra 18-vs-18 random comparison with same 100 seeds (exploratory strict sensitivity).
    strict_random_rows=[]
    for repeat in range(100):
        comp=strict_linkable.sample(n=18,replace=False,random_state=CV_SEED+repeat)
        _,res=rq1_semantic(rq1_pair(strict_positives,comp),'strict18_repeated_random','all_posts')
        strict_random_rows.append({'repeat':repeat+1,'random_seed':CV_SEED+repeat,**res})
    atomic_csv(pd.DataFrame(strict_random_rows),RQ1/'RQ1_strict18_random_100_repetitions.csv')
    print('Repeated random sensitivity PASS; FDR PC counts:',
          RQ1_RANDOM_REPEATS.n_PCs_FDR_significant.describe().to_dict())
else:
    print('Repeated random sensitivity PENDING — set RUN_RQ1_REPEATED_RANDOM=True.')

In [ ]:
# CELL 11 — R1-LOCKED PRETRAINED MENTALBERT CLASSIFICATION
# ORIGINAL MAJOR-REVISION METHOD: PCA -> StandardScaler -> Logistic Regression
# PCA and scaler are FIT ONLY ON TRAINING FOLDS (no held-out leakage).
# Output namespace is NEW; existing StandardScaler->PCA results are preserved.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from pathlib import Path
import numpy as np
import pandas as pd
import gc

R1LOCKED_PRETRAINED_METRICS = (
    BASE / 'R2_pretrained_MentalBERT_PCA_then_scaler_R1LOCKED_metrics.csv'
)
R1LOCKED_PRETRAINED_OOF = (
    BASE / 'R2_pretrained_MentalBERT_PCA_then_scaler_R1LOCKED_OOF_PRIVATE.csv'
)

PRETRAINED_METRICS = []
PRETRAINED_OOF = []
for task, frame in DATASETS.items():
    ix = np.asarray([lookup[str(pid)] for pid in frame.post_id], dtype=int)
    X = np.asarray(embeddings[ix], dtype=np.float32)
    y = frame.label.to_numpy(dtype=int)
    groups = frame.analysis_group.astype(str).to_numpy()
    prob = np.full(len(y), np.nan)
    folds = np.asarray(ALL_FOLDS[task])
    assert len(folds) == len(frame)

    for fold_number, (tr, te) in enumerate(ALL_SPLITS[task], 1):
        assert len(set(y[tr])) == 2 and len(set(y[te])) == 2
        assert set(groups[tr]).isdisjoint(set(groups[te])), (
            f'Author-group leakage: {task}, fold {fold_number}'
        )
        k = min(50, len(tr) - 2, X.shape[1])
        assert k >= 2
        pipe = Pipeline([
            ('pca', PCA(
                n_components=k,
                svd_solver='randomized',
                random_state=42
            )),
            ('standardize', StandardScaler()),
            ('lr', LogisticRegression(
                class_weight='balanced',
                solver='liblinear',
                max_iter=3000,
                random_state=42
            ))
        ])
        pipe.fit(X[tr], y[tr])
        prob[te] = pipe.predict_proba(X[te])[:, 1]

    assert np.isfinite(prob).all(), task
    result = evaluate_predictions(y, prob)
    PRETRAINED_METRICS.append({
        'scenario': task,
        'model': 'MentalBERT_PCA50_LR',
        'n': len(frame),
        'positives': int(y.sum()),
        **result
    })
    PRETRAINED_OOF.append(pd.DataFrame({
        'scenario': task,
        'post_id': frame.post_id.astype(str),
        'label': y,
        'analysis_group': groups,
        'cv_fold': folds,
        'score': prob
    }))
    print('R1-LOCKED PRETRAINED', task, 'AUC', round(result['ROC_AUC'], 4))
    del X
    gc.collect()

PRETRAINED_METRICS = pd.DataFrame(PRETRAINED_METRICS)
PRETRAINED_OOF = pd.concat(PRETRAINED_OOF, ignore_index=True)
assert PRETRAINED_METRICS.scenario.nunique() == len(DATASETS) == 10
assert PRETRAINED_OOF.groupby('scenario').post_id.nunique().eq(
    PRETRAINED_OOF.groupby('scenario').size()
).all()

atomic_csv(PRETRAINED_METRICS, R1LOCKED_PRETRAINED_METRICS)
atomic_csv(PRETRAINED_OOF, R1LOCKED_PRETRAINED_OOF)
print('CELL 11 R1-LOCKED PASS — PCA -> StandardScaler, fold-local, randomized solver')
print('Metrics:', R1LOCKED_PRETRAINED_METRICS)
print('OOF:', R1LOCKED_PRETRAINED_OOF)


In [ ]:
# 12 — Paired author-group bootstrap delta for pretrained vs raw/masked TF-IDF.
# Deltas are LEFT minus RIGHT; the same posts and fold assignments are required.
def paired_group_delta(left,right,name,n_boot=N_BOOT):
    req=['post_id','analysis_group','label','cv_fold','score']
    assert set(req)<=set(left.columns) and set(req)<=set(right.columns)
    a=left[req].copy().rename(columns={'score':'left_score'})
    b=right[req].copy().rename(columns={'score':'right_score','analysis_group':'group_b',
        'label':'label_b','cv_fold':'fold_b'})
    assert a.post_id.is_unique and b.post_id.is_unique
    j=a.merge(b,on='post_id',validate='one_to_one')
    assert len(j)==len(a)==len(b)
    assert (j.label==j.label_b).all() and (j.analysis_group==j.group_b).all()
    assert (j.cv_fold==j.fold_b).all(),'Fold mismatch: models are not directly paired.'
    y=j.label.to_numpy(int);pl=j.left_score.to_numpy(float);pr=j.right_score.to_numpy(float)
    assert np.isfinite(pl).all() and np.isfinite(pr).all()
    unique,inv=np.unique(j.analysis_group.astype(str),return_inverse=True)
    group_idx=[np.flatnonzero(inv==i) for i in range(len(unique))]
    rng=np.random.default_rng(2026)
    delta_auc=[];delta_ap=[]
    for _ in range(n_boot):
        draw=rng.integers(len(unique),size=len(unique))
        ix=np.concatenate([group_idx[i] for i in draw])
        if len(set(y[ix]))<2:continue
        delta_auc.append(roc_auc_score(y[ix],pl[ix])-roc_auc_score(y[ix],pr[ix]))
        delta_ap.append(average_precision_score(y[ix],pl[ix])-average_precision_score(y[ix],pr[ix]))
    assert len(delta_auc)>=int(n_boot*.95)
    return {'comparison':name,'n':len(y),'author_groups':len(unique),
        'delta_ROC_AUC':roc_auc_score(y,pl)-roc_auc_score(y,pr),
        'delta_AUC_CI_low':float(np.quantile(delta_auc,.025)),
        'delta_AUC_CI_high':float(np.quantile(delta_auc,.975)),
        'delta_AP':average_precision_score(y,pl)-average_precision_score(y,pr),
        'delta_AP_CI_low':float(np.quantile(delta_ap,.025)),
        'delta_AP_CI_high':float(np.quantile(delta_ap,.975)),
        'valid_bootstrap_samples':len(delta_auc)}

def base_slice(task,model):
    return BASE_OOF.loc[(BASE_OOF.scenario==task)&(BASE_OOF.model==model)].copy()

def pretrained_slice(task):
    return PRETRAINED_OOF.loc[PRETRAINED_OOF.scenario==task].copy()

PRETRAINED_PAIRS=[]
for task in ('rq2_primary_58v116','rq2_primary_58v116_hardneg',
             'rq2_candidate_58v300','rq2_tight_58v58',
             'rq2_strict18v36','rq2_strict18v36_hardneg',
             'rq2_strict18v300','rq2_strict18v18'):
    for comparator in ('tfidf_raw','tfidf_masked'):
        PRETRAINED_PAIRS.append({'scenario':task,**paired_group_delta(
            pretrained_slice(task),base_slice(task,comparator),
            'Pretrained_MentalBERT_minus_'+comparator)})
PRETRAINED_DELTA=pd.DataFrame(PRETRAINED_PAIRS)
atomic_csv(PRETRAINED_DELTA,BASE/'R2_pretrained_MentalBERT_paired_author_bootstrap.csv')
print(PRETRAINED_DELTA.round(4).to_string(index=False))

## Fine-tuned MentalBERT — eight prespecified tasks, 240 grouped fits

The eight tasks are: primary 58:116, primary 58:116 reviewed hard negatives,
primary 58:300 reviewed hard negatives, primary 58:58 tight length-match,
strict 18:36, strict 18:36 reviewed hard negatives, strict 18:300 reviewed
hard negatives, and strict 18:18 tight length-match. Each runs raw and masked
text, three independent seeds (42,123,2026), and five author-disjoint folds.

Historical R1 training protocol: AdamW, constant learning rate 2e-5, 2 epochs,
512 tokens, batch 4, gradient accumulation 2, mixed precision. The 18-v2 labels
remain post-hoc exploratory until separately reconfirmed. Seven of 18 posts
share one author group; report per-seed and grouped uncertainty.


In [ ]:

# ============================================================
# CELL 13 — FREEZE MENTALBERT FINE-TUNING CONFIGURATION
# ============================================================
#
# Purpose:
# Define the eight prespecified MentalBERT fine-tuning
# scenarios and freeze all training parameters,
# author-grouped fold assignments, model revision,
# input checksums, and random seeds.
#
# Methods:
# - Fine-tuned MentalBERT
# - Five author-grouped cross-validation folds
# - Three training seeds: 42, 123, 2026
# - Raw and broadly keyword-masked text
# - Fixed learning rate: 2e-5
# - Two training epochs
# - Maximum sequence length: 512 tokens
# - Effective batch size: 8
#
# The same splits are used for raw/masked comparisons
# and across the three fine-tuning seeds.
#
# Total planned training jobs:
# 8 scenarios x 2 representations x 3 seeds x 5 folds
# = 240 fold-level fine-tuning runs.
#
# Important:
# No model selection or hyperparameter tuning is
# performed using held-out fold performance.
# ============================================================

from transformers import AutoTokenizer

import hashlib
import json


# ------------------------------------------------------------
# 1. PRESPECIFIED FINE-TUNING SCENARIOS
# ------------------------------------------------------------

FT_TASKS = [

    # Primary 1:2 comparison:
    # 58 positive vs 116 community comparison posts.
    "rq2_primary_58v116",

    # Primary 1:2 hard-negative challenge:
    # 58 positive vs 116 manually reviewed Excludes.
    "rq2_primary_58v116_hardneg",

    # Strict high-specificity 1:2 comparison:
    # 18 positive vs 36 community comparison posts.
    "rq2_strict18v36",

    # Strict 1:2 hard-negative challenge:
    # 18 positive vs 36 reviewed Excludes.
    "rq2_strict18v36_hardneg",

    # Tight length-matched 1:1 comparisons.
    "rq2_tight_58v58",
    "rq2_strict18v18",

    # Full manually reviewed hard-negative challenges.
    "rq2_candidate_58v300",
    "rq2_strict18v300"
]

assert len(FT_TASKS) == 8

assert all(
    task in DATASETS
    for task in FT_TASKS
)

assert all(
    task in ALL_SPLITS
    for task in FT_TASKS
)


# ------------------------------------------------------------
# 2. TEXT REPRESENTATIONS
# ------------------------------------------------------------

FT_REPS = [
    "raw",
    "broad_masked"
]

assert len(FT_REPS) == 2

assert N_FOLDS == 5
assert SEEDS == [42, 123, 2026]
assert MAX_TOKENS == 512

assert MAX_FT_EPOCHS == 2
assert FT_BATCH == 4
assert FT_ACCUM == 2
assert FT_LR == 2e-5


# ------------------------------------------------------------
# 3. FINE-TUNING OUTPUT DIRECTORY
# ------------------------------------------------------------

FT_RUN = (
    FT /
    "DUAL_58_18_v6_FINAL_NO415_PINNED_R1constantLR"
)

FT_RUN.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 4. FREEZE TRAINING CONFIGURATION
# ------------------------------------------------------------

TRAINING_MANIFEST = {

    "label_sha256": INPUT_SHA,

    "master_sha256": MASTER_SHA,

    "task_dataset_sha256": {
        task: sha256_file(
            DATA / (task + "_PRIVATE.csv")
        )
        for task in FT_TASKS
    },

    "model": MODEL_NAME,

    "revision": MODEL_REVISION,

    "max_tokens": MAX_TOKENS,

    "seeds": SEEDS,

    "cv_split_seed_start": CV_SEED,

    "cv_split_policy": (
        "First class-complete StratifiedGroupKFold "
        "seed selected from a deterministic sequence. "
        "No performance-based split selection."
    ),

    "cv_split_selected_seed_by_task": (
        CV_SELECTED_SEEDS
    ),

    "n_folds": N_FOLDS,

    "epochs": MAX_FT_EPOCHS,

    "batch_size": FT_BATCH,

    "gradient_accumulation": FT_ACCUM,

    "effective_batch_size": (
        FT_BATCH * FT_ACCUM
    ),

    "learning_rate": FT_LR,

    "training_protocol": FT_TRAINING_PROTOCOL,

    "mask_regex": BROAD.pattern,

    "fold_assignment_sha256": {
        task: hashlib.sha256(
            ALL_FOLDS[task].tobytes()
        ).hexdigest()
        for task in FT_TASKS
    },

    "strict_v2_status": (
        "Posthoc proposed high-specificity subset "
        "pending independent strict confirmation "
        "of two replacements."
    ),

    "status": (
        "Primary 58 labels adjudicated; "
        "strict 18 v2 proposed; "
        "no threshold or model selection "
        "on out-of-fold scores."
    )
}


# ------------------------------------------------------------
# 5. REPRODUCIBILITY / CACHE SAFETY
# ------------------------------------------------------------

manifest_file = (
    FT_RUN / "training_manifest.json"
)

if manifest_file.exists():

    previous_manifest = json.loads(
        manifest_file.read_text(
            encoding="utf-8"
        )
    )

    assert previous_manifest == TRAINING_MANIFEST, (
        "Existing fine-tuning cache belongs to "
        "a different configuration. "
        "Do not overwrite it. "
        "Use a new FT_RUN directory if the "
        "training configuration has changed."
    )

else:

    save_json(
        manifest_file,
        TRAINING_MANIFEST
    )


# ------------------------------------------------------------
# 6. LOAD PINNED MENTALBERT TOKENIZER
# ------------------------------------------------------------

TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    use_fast=True
)


# ------------------------------------------------------------
# 7. FINAL CONFIGURATION AUDIT
# ------------------------------------------------------------

N_FT_JOBS = (
    len(FT_TASKS)
    * len(FT_REPS)
    * len(SEEDS)
    * N_FOLDS
)

assert N_FT_JOBS == 240

print("=" * 65)
print("MENTALBERT FINE-TUNING CONFIGURATION: PASS")
print("=" * 65)

print("Model:", MODEL_NAME)
print("Revision:", MODEL_REVISION)
print("Max tokens:", MAX_TOKENS)
print("Learning rate:", FT_LR)
print("Epochs:", MAX_FT_EPOCHS)
print("Effective batch size:", FT_BATCH * FT_ACCUM)

print("Fine-tuning scenarios:", len(FT_TASKS))
print("Text representations:", FT_REPS)
print("Training seeds:", SEEDS)
print("Grouped CV folds:", N_FOLDS)

print("Total planned training jobs:", N_FT_JOBS)

print("\nSelected CV split seeds:")

for task in FT_TASKS:
    print(
        task,
        "=>",
        CV_SELECTED_SEEDS[task]
    )

print("\nManifest saved:", manifest_file)
print("Fine-tuning directory:", FT_RUN)

print(
    "\nNOTE: Strict 18 v2 human reconfirmation "
    "remains pending."
)

print(
    "\nCELL 13 COMPLETE — "
    "Training configuration frozen."
)


In [ ]:
# 14 — GPU training helper; all fold predictions are resumable and validated.
from torch.utils.data import Dataset,DataLoader
from transformers import AutoModelForSequenceClassification,DataCollatorWithPadding

class BinaryTextDataset(Dataset):
    def __init__(self,texts,labels=None):
        self.encoded=TOKENIZER(list(map(str,texts)),truncation=True,
            max_length=MAX_TOKENS,padding=False)
        self.labels=None if labels is None else list(map(int,labels))
    def __len__(self):return len(self.encoded['input_ids'])
    def __getitem__(self,i):
        item={k:v[i] for k,v in self.encoded.items()}
        if self.labels is not None:item['labels']=self.labels[i]
        return item

FT_COLLATOR=DataCollatorWithPadding(tokenizer=TOKENIZER)

def ft_fold_path(task,rep,seed,fold):
    return FT_RUN/f'{task}__{rep}__seed{seed}__fold{fold}.csv'

def read_ft_fold(task,rep,seed,fold):
    path=ft_fold_path(task,rep,seed,fold)
    assert path.exists(),f'Missing OOF fold {path.name}'
    z=pd.read_csv(path,dtype={'post_id':str})
    _,te=ALL_SPLITS[task][fold-1]
    expected=DATASETS[task].iloc[te]
    assert len(z)==len(expected)
    assert z.post_id.tolist()==expected.post_id.tolist()
    assert z.label.tolist()==expected.label.astype(int).tolist()
    assert z.analysis_group.tolist()==expected.analysis_group.astype(str).tolist()
    assert z.fold.eq(fold).all() and z.seed.eq(seed).all()
    assert z.scenario.eq(task).all() and z.representation.eq(rep).all()
    assert np.isfinite(z.score).all() and z.score.between(0,1).all()
    return z

def set_training_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def fit_mentalbert(train_texts,train_labels,seed,save_model_dir=None):
    assert torch.cuda.is_available(),'Enable T4 GPU before fine-tuning.'
    assert FT_TRAINING_PROTOCOL=='r1_constant_lr_no_scheduler_no_clip_amp'
    set_training_seed(seed)
    ds=BinaryTextDataset(train_texts,train_labels)
    generator=torch.Generator().manual_seed(seed)
    loader=DataLoader(ds,batch_size=FT_BATCH,shuffle=True,collate_fn=FT_COLLATOR,
        generator=generator,num_workers=0)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,revision=MODEL_REVISION,num_labels=2).to('cuda')
    optim=torch.optim.AdamW(model.parameters(),lr=FT_LR)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    for epoch in range(MAX_FT_EPOCHS):
        model.train();optim.zero_grad(set_to_none=True)
        accumulated=0.0
        for j,batch in enumerate(loader,1):
            batch={k:v.to('cuda') for k,v in batch.items()}
            with torch.autocast('cuda',dtype=torch.float16):
                out=model(**batch);loss=out.loss/FT_ACCUM
            scaler.scale(loss).backward()
            accumulated+=float(out.loss.detach().cpu())
            if j%FT_ACCUM==0 or j==len(loader):
                scaler.step(optim)
                scaler.update()
                optim.zero_grad(set_to_none=True)
        print('seed',seed,'epoch',epoch+1,'loss',round(accumulated/len(loader),4))
    if save_model_dir is not None:
        save_model_dir.mkdir(parents=True,exist_ok=True)
        model.save_pretrained(save_model_dir,safe_serialization=True)
        TOKENIZER.save_pretrained(save_model_dir)
    del optim,scaler,ds,loader
    return model

@torch.inference_mode()
def mentalbert_predict(model,texts):
    ds=BinaryTextDataset(texts)
    loader=DataLoader(ds,batch_size=FT_BATCH,shuffle=False,
        collate_fn=FT_COLLATOR,num_workers=0)
    model.eval();prob=[]
    for batch in loader:
        batch={k:v.to('cuda') for k,v in batch.items()}
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(**batch).logits
        prob.extend(torch.softmax(logits.float(),dim=-1)[:,1].cpu().tolist())
    return np.asarray(prob,float)

def run_ft_fold(task,rep,seed,fold):
    assert task in FT_TASKS and rep in FT_REPS and seed in SEEDS and fold in range(1,6)
    dest=ft_fold_path(task,rep,seed,fold)
    if dest.exists():
        cached=read_ft_fold(task,rep,seed,fold)
        print('CACHED',dest.name,'n',len(cached));return cached
    data=DATASETS[task];tr,te=ALL_SPLITS[task][fold-1]
    xtr=data.iloc[tr].text.tolist();xte=data.iloc[te].text.tolist()
    if rep=='broad_masked':
        xtr=[mask_text(x) for x in xtr];xte=[mask_text(x) for x in xte]
    model=fit_mentalbert(xtr,data.iloc[tr].label.to_numpy(int),seed)
    scores=mentalbert_predict(model,xte)
    assert len(scores)==len(te) and np.isfinite(scores).all()
    z=pd.DataFrame({'scenario':task,'representation':rep,'seed':seed,'fold':fold,
        'post_id':data.iloc[te].post_id.tolist(),
        'analysis_group':data.iloc[te].analysis_group.tolist(),
        'label':data.iloc[te].label.to_numpy(int),'score':scores})
    atomic_csv(z,dest)
    del model;gc.collect();torch.cuda.empty_cache()
    print('SAVED',dest.name,'fold AUC',round(roc_auc_score(z.label,z.score),4))
    return read_ft_fold(task,rep,seed,fold)

print('FINE-TUNING FUNCTIONS READY; no fold trained by this definition cell.')

In [ ]:
# 15 — One-fold GPU smoke test; cached during the complete run.
if RUN_FINE_TUNING:
    probe=run_ft_fold('rq2_candidate_58v300','raw',42,1)
    print('SMOKE TEST PASS',len(probe),'OOF posts','AUC',round(roc_auc_score(probe.label,probe.score),4))
else:
    print('GPU training not requested. Fine-tuning remains PENDING.')

In [ ]:

# ============================================================
# CELL 16 — FULL RAW MENTALBERT FINE-TUNING
# ============================================================
#
# Purpose:
# Execute all prespecified MentalBERT fine-tuning runs
# using the original, unmasked Reddit post text.
#
# Experimental design:
# - 8 classification scenarios
# - 3 independent training seeds: 42, 123, 2026
# - 5 author-grouped cross-validation folds
# - Total: 120 RAW fine-tuning runs
#
# Resume policy:
# - Each completed fold is saved to Google Drive as CSV.
# - Existing fold files are validated before reuse.
# - Completed folds are not retrained.
# - Interrupted or missing folds are trained on the next run.
#
# The same frozen folds will be reused in Cell 17
# for the broad-masked MentalBERT experiments.
# ============================================================

import torch

assert RUN_FINE_TUNING, (
    "RUN_FINE_TUNING must be True."
)

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable a Colab GPU runtime."
)

assert len(FT_TASKS) == 8
assert len(SEEDS) == 3
assert N_FOLDS == 5

# Build the complete list of expected RAW training jobs.
raw_jobs = [
    (task, seed, fold)
    for task in FT_TASKS
    for seed in SEEDS
    for fold in range(1, N_FOLDS + 1)
]

assert len(raw_jobs) == 120

# Check existing fold files before starting training.
cached_before = 0

for task, seed, fold in raw_jobs:

    if ft_fold_path(
        task, "raw", seed, fold
    ).exists():

        # Validate the existing cached predictions.
        read_ft_fold(task, "raw", seed, fold)

        cached_before += 1

print("=" * 65)
print("RAW MENTALBERT FINE-TUNING")
print("=" * 65)

print("GPU:", torch.cuda.get_device_name(0))
print("Total expected RAW jobs:", len(raw_jobs))
print("Validated cached jobs:", cached_before)
print("Remaining jobs:", len(raw_jobs) - cached_before)

print("=" * 65)

# Run missing folds and reuse validated completed folds.
for job_number, (task, seed, fold) in enumerate(
    raw_jobs, start=1
):

    print(
        f"\nRAW JOB {job_number}/{len(raw_jobs)}"
        f" | {task}"
        f" | seed={seed}"
        f" | fold={fold}",
        flush=True
    )

    run_ft_fold(
        task,
        "raw",
        seed,
        fold
    )

# Final audit: all 120 fold files must exist and validate.
for task, seed, fold in raw_jobs:

    read_ft_fold(
        task,
        "raw",
        seed,
        fold
    )

print("\n" + "=" * 65)
print("ALL RAW FOLDS PASS")
print("Completed RAW jobs:", len(raw_jobs))
print("Fine-tuning tasks:", len(FT_TASKS))
print("Training seeds:", len(SEEDS))
print("Cross-validation folds:", N_FOLDS)
print("=" * 65)


In [ ]:

# ============================================================
# CELL 17 — FULL BROAD-MASKED MENTALBERT FINE-TUNING
# ============================================================
#
# Purpose:
# Repeat the MentalBERT fine-tuning experiments using
# broadly keyword-masked Reddit post text.
#
# Experimental design:
# - 8 prespecified classification scenarios
# - 3 training seeds
# - 5 author-grouped cross-validation folds
# - Total: 120 MASKED training jobs
#
# Reproducibility:
# - Use exactly the same frozen folds as RAW training.
# - Use the same learning rate, epochs, and batch size.
# - Save each completed fold separately to Google Drive.
# - Validate and reuse existing completed fold files.
#
# These experiments assess model sensitivity to explicit
# addiction-related vocabulary.
# ============================================================

import torch

assert RUN_FINE_TUNING
assert torch.cuda.is_available()
assert len(FT_TASKS) == 8
assert len(SEEDS) == 3
assert N_FOLDS == 5
assert "broad_masked" in FT_REPS

masked_jobs = [
    (task, seed, fold)
    for task in FT_TASKS
    for seed in SEEDS
    for fold in range(1, N_FOLDS + 1)
]

assert len(masked_jobs) == 120

# Identify and validate previously completed jobs.
cached_before = 0

for task, seed, fold in masked_jobs:
    if ft_fold_path(
        task, "broad_masked", seed, fold
    ).exists():
        read_ft_fold(
            task, "broad_masked", seed, fold
        )
        cached_before += 1

print("=" * 65)
print("BROAD-MASKED MENTALBERT FINE-TUNING")
print("=" * 65)
print("GPU:", torch.cuda.get_device_name(0))
print("Expected MASKED jobs:", len(masked_jobs))
print("Validated cached jobs:", cached_before)
print("Remaining jobs:", len(masked_jobs) - cached_before)

# Train missing folds and reuse validated cached predictions.
for job_number, (task, seed, fold) in enumerate(
    masked_jobs, start=1
):
    print(
        f"\nMASKED JOB {job_number}/{len(masked_jobs)}"
        f" | {task}"
        f" | seed={seed}"
        f" | fold={fold}",
        flush=True
    )

    run_ft_fold(
        task,
        "broad_masked",
        seed,
        fold
    )

# Verify that every expected MASKED fold is complete.
for task, seed, fold in masked_jobs:
    read_ft_fold(
        task,
        "broad_masked",
        seed,
        fold
    )

print("\n" + "=" * 65)
print("ALL MASKED FOLDS PASS")
print("Completed MASKED jobs:", len(masked_jobs))
print("Combined RAW + MASKED jobs:", 240)
print("=" * 65)


In [ ]:
# 18 — Three-seed OOF metrics. Report independent seeds as PRIMARY,
# and the arithmetic mean of seed probabilities only as a SECONDARY ensemble.
if RUN_FINE_TUNING:
    SEED_METRICS=[];ENSEMBLE={};FT_SEED_OOF={}
    for task in FT_TASKS:
        data=DATASETS[task]
        for rep in FT_REPS:
            scores=[]
            for seed in SEEDS:
                z=pd.concat([read_ft_fold(task,rep,seed,k) for k in range(1,6)],ignore_index=True)
                z=data[['post_id','label','analysis_group']].assign(cv_fold=ALL_FOLDS[task]).merge(
                    z[['post_id','score','fold']],on='post_id',validate='one_to_one')
                assert len(z)==len(data) and (z.cv_fold==z.fold).all() and z.score.notna().all()
                scores.append(z.score.to_numpy(float))
                FT_SEED_OOF[(task,rep,seed)]=z[['post_id','label','analysis_group','cv_fold','score']].copy()
                m=evaluate_predictions(z.label,z.score)
                SEED_METRICS.append({'scenario':task,'representation':rep,'seed':seed,
                    'n':len(z),'positives':int(z.label.sum()),**m})
                atomic_csv(z,FT_RUN/f'{task}__{rep}__seed{seed}__ALL_OOF_PRIVATE.csv')
            ens=np.mean(np.stack(scores),axis=0)
            out=data[['post_id','label','analysis_group']].assign(cv_fold=ALL_FOLDS[task],score=ens)
            ENSEMBLE[(task,rep)]=out
            atomic_csv(out,FT_RUN/f'{task}__{rep}__MEAN_3SEEDS_OOF_PRIVATE.csv')
    FT_METRICS=pd.DataFrame(SEED_METRICS)
    atomic_csv(FT_METRICS,FT_RUN/'R2_finetuned_MentalBERT_per_seed_metrics.csv')
    FT_SUMMARY=FT_METRICS.groupby(['scenario','representation'],as_index=False).agg(
        auc_mean=('ROC_AUC','mean'),auc_sd=('ROC_AUC','std'),ap_mean=('AP','mean'),ap_sd=('AP','std'),
        bacc_mean=('BAcc','mean'),f1_mean=('F1','mean'))
    atomic_csv(FT_SUMMARY,FT_RUN/'R2_finetuned_MentalBERT_three_seed_summary.csv')
    print('PRIMARY THREE-SEED SUMMARY:')
    print(FT_SUMMARY.round(4).to_string(index=False))
    print('SECONDARY ENSEMBLE:')
    for (task,rep),z in ENSEMBLE.items():
        print(task,rep,evaluate_predictions(z.label,z.score)['ROC_AUC'])
else:
    print('C18 PENDING: run 300 fine-tuning folds before computing a final summary.')

In [ ]:
# 19 — Group-clustered paired bootstrap on SECONDARY three-seed OOF ensembles.
# All comparisons are fold-aligned, and the model development is nested inside training folds.
if RUN_FINE_TUNING:
    rows=[]
    for task in FT_TASKS:
        for rep,comparison in [('raw','tfidf_raw'),('broad_masked','tfidf_masked')]:
            delta=paired_group_delta(ENSEMBLE[(task,rep)],base_slice(task,comparison),
                'FineTuned_'+rep+'_minus_'+comparison)
            rows.append({'scenario':task,**delta})
        rows.append({'scenario':task,**paired_group_delta(ENSEMBLE[(task,'raw')],
            ENSEMBLE[(task,'broad_masked')],'FineTuned_raw_minus_FineTuned_masked')})
    FT_BOOTSTRAP=pd.DataFrame(rows)
    atomic_csv(FT_BOOTSTRAP,FT_RUN/'R2_finetuned_ensemble_paired_bootstrap_5000.csv')
    print(FT_BOOTSTRAP.round(4).to_string(index=False))
    print('Secondary ensemble CIs only; multiple exploratory comparisons not adjusted.')
else:print('C19 PENDING')

In [ ]:
# 20 — Per-seed baseline comparisons and raw-vs-masked stability (no retraining).
if RUN_FINE_TUNING:
    stability=[]
    for task in FT_TASKS:
        for rep,comparison in [('raw','tfidf_raw'),('broad_masked','tfidf_masked')]:
            baseline=base_slice(task,comparison)
            for seed in SEEDS:
                ft=FT_SEED_OOF[(task,rep,seed)]
                join=ft.merge(baseline[['post_id','score','cv_fold','label','analysis_group']],
                    on='post_id',validate='one_to_one',suffixes=('_ft','_baseline'))
                assert len(join)==len(ft)==len(baseline)
                assert (join.label_ft==join.label_baseline).all()
                assert (join.cv_fold_ft==join.cv_fold_baseline).all()
                assert (join.analysis_group_ft==join.analysis_group_baseline).all()
                stability.append({'scenario':task,'representation':rep,'seed':seed,
                    'delta_AUC':roc_auc_score(join.label_ft,join.score_ft)-roc_auc_score(join.label_ft,join.score_baseline),
                    'delta_AP':average_precision_score(join.label_ft,join.score_ft)-average_precision_score(join.label_ft,join.score_baseline)})
    FT_STABILITY=pd.DataFrame(stability)
    atomic_csv(FT_STABILITY,FT_RUN/'R2_finetuned_per_seed_vs_TFIDF.csv')
    print(FT_STABILITY.round(4).to_string(index=False))
else:print('C20 PENDING')

# 20B — Second-round MINOR-reviewer comparison on the EXACT SAME hard-negative rows.
# The previously published 0.619 (MentalBERT) and 0.658 (keyword rule) refer to
# the old 197-positive task; DO NOT carry those numbers into 58-/18-post R2.
if RUN_FINE_TUNING:
    contrast=[]
    for task in ('rq2_primary_58v116_hardneg','rq2_candidate_58v300',
                 'rq2_strict18v36_hardneg','rq2_strict18v300'):
        rule=base_slice(task,'strict5_fixed_rule').copy().set_index('post_id')
        for seed in SEEDS:
            ft=FT_SEED_OOF[(task,'raw',seed)].copy()
            assert len(ft)==len(rule) and set(ft.post_id)==set(rule.index)
            rule=rule.loc[ft.post_id.astype(str)]
            assert np.array_equal(ft.label.to_numpy(int),rule.label.to_numpy(int))
            kw_auc=roc_auc_score(ft.label,rule.score)
            mental_auc=roc_auc_score(ft.label,ft.score)
            contrast.append({'task':task,'seed':seed,
                'n_positive':int(ft.label.sum()),'n_negative':int(len(ft)-ft.label.sum()),
                'mentalbert_raw_AUC':mental_auc,'keyword_presence_AUC':kw_auc,
                'mentalbert_minus_keyword_AUC':mental_auc-kw_auc,
                'mentalbert_raw_AP':average_precision_score(ft.label,ft.score),
                'keyword_presence_AP':average_precision_score(ft.label,rule.score)})
    minor_contrast=pd.DataFrame(contrast)
    atomic_csv(minor_contrast,MANUSCRIPT/'MINOR_reviewer_same_task_MentalBERT_vs_keyword_rule.csv')
    print('REVIEWER 2/4 SAME-TASK HARD-NEGATIVE COMPARISON:')
    print(minor_contrast.round(4).to_string(index=False))


In [ ]:
# 21 — Private error analysis: OOF misclassifications for the hard-negative challenge.
# For reviewer discussion only; never publish identifiable texts or clinical labels.
if RUN_FINE_TUNING:
    task='rq2_candidate_58v300'
    raw=ENSEMBLE[(task,'raw')].copy().rename(columns={'score':'fine_tuned_raw'})
    masked=ENSEMBLE[(task,'broad_masked')][['post_id','score']].rename(columns={'score':'fine_tuned_masked'})
    tfidf=base_slice(task,'tfidf_raw')[['post_id','score']].rename(columns={'score':'tfidf_raw'})
    errors=raw.merge(masked,on='post_id',validate='one_to_one').merge(tfidf,on='post_id',validate='one_to_one')
    errors=errors.merge(labels[['post_id','audit_id','spiros','daniel','vas','final_decision','vas_notes']],
        on='post_id',validate='one_to_one')
    errors['finetuned_raw_prediction']=(errors.fine_tuned_raw>=.5).astype(int)
    errors['finetuned_raw_misclassified']=errors.finetuned_raw_prediction.ne(errors.label)
    errors['reviewer_disagreement']=errors.spiros.ne(errors.daniel)
    atomic_csv(errors,AUDIT/'R2_hard_negative_errors_three_reviewer_PRIVATE.csv')
    print('Misclassified hard-negative-challenge posts:',int(errors.finetuned_raw_misclassified.sum()))
    print('Of which previously disputed:',int((errors.finetuned_raw_misclassified&errors.reviewer_disagreement).sum()))
    print('These scores are not grounds to change adjudicated human labels.')
else:print('C21 PENDING')

## Label provenance and external-application gates

The revised high-specificity v2 **retains 16 historical strict posts and swaps in 2 primary-confirmed positives**. Treat results as exploratory/post-hoc until independent strict adjudication of the two additions. Never claim the model independently validated this human-derived selection. The old outside-keyword 300-post review and old 36 DDO/manual WoW reviews **do not** become valid labels for new scores. External corpora remain unlabeled during model development; manual review occurs only *after* frozen scoring. No external threshold tuning.


In [ ]:
# 22A — Verify all dual-label model scenarios have been constructed and evaluated.
assert set(STRICT_DATASETS).issubset(set(DATASETS))
assert all(k in ALL_SPLITS for k in STRICT_DATASETS)
for name in STRICT_DATASETS:
    assert BASE_METRICS.scenario.eq(name).any(),f'Missing strict classical metrics for {name}'
    assert PRETRAINED_METRICS.scenario.eq(name).any(),f'Missing strict pretrained metrics for {name}'
strict_descriptive=length_audit[length_audit.scenario.isin(STRICT_DATASETS)]
atomic_csv(strict_descriptive,MANUSCRIPT/'strict18_v2_scenario_length_diagnostics.csv')
print('STRICT 18 V2: all three grouped-CV + pretrained baseline tasks PASS.')
print('Computational sensitivity checks completed; human coding is reported separately in the manuscript.')


## External candidate selection — optional, NOT external validation

Once **complete** DDO 948 and (optionally) WoW 265,009 eligible corpora are
available, freeze both trained cohorts (58:116 and 18:36) and score every post
using logistic regression, SVM, random forest and the mean of three fine-tuned
MentalBERT seeds. Raw/masked variants and consensus thresholds ≥0.60, ≥0.70
remain fixed. No DDO/WoW human labels are used for model training or tuning.

The old 36 DDO IDs and 5,243 WoW IDs are optional historical overlap references,
not model inputs or replacement corpora. Model-positive candidates and a random
DDO unselected sample must be reviewed blindly; yield on selected posts alone
does not estimate recall, specificity, prevalence, diagnosis or clinical accuracy.


In [ ]:

from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/"
    "MMORPG_DigitalHealth_R2_58_18_v6_FINAL_NO415/"
    "00_inputs"
)

source = folder / "WOW_197_main_strict_three_seed_predictions.parquet"

wow = pd.read_parquet(
    source,
    columns=["wow_post_id", "title_std", "body_std"]
)

wow["post_id"] = wow["wow_post_id"].astype(str).str.strip()

wow["full_text"] = (
    wow["title_std"].fillna("").astype(str)
    + " "
    + wow["body_std"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

wow = wow[["post_id", "full_text"]]

assert len(wow) == 265009
assert wow.post_id.is_unique
assert wow.full_text.str.len().ge(40).all()

destination = folder / "WoW_eligible_PRIVATE.csv"
wow.to_csv(destination, index=False)

print("WoW dataset ready:", len(wow))
print("Saved:", destination)


In [ ]:
%pip install -q odfpy

# DDO — Convert original ODS to the canonical 948-post CSV.
# Reproduce the original eligibility and stable-ID rules.

from pathlib import Path
import pandas as pd
import hashlib

folder = Path(
    "/content/drive/MyDrive/"
    "MMORPG_DigitalHealth_R2_58_18_v6_FINAL_NO415/"
    "00_inputs"
)

source = folder / "ddo_posts_top.ods"
destination = folder / "DDO_eligible_948_PRIVATE.csv"

assert source.is_file(), f"Missing source: {source}"

raw = pd.read_excel(
    source,
    sheet_name="ddo_all_threads_top",
    engine="odf"
)

assert len(raw) == 1000
assert {"url", "title", "text"}.issubset(raw.columns)

def clean(col):
    return raw[col].fillna("").astype(str).str.strip()

urls = clean("url")
titles = clean("title")
bodies = clean("text")

ddo = pd.DataFrame({
    "post_id": urls.map(
        lambda x: "ddo_" + hashlib.sha256(
            x.encode("utf-8")
        ).hexdigest()[:20]
    ),
    "full_text": (
        titles + " " + bodies
    ).str.replace(r"\s+", " ", regex=True).str.strip()
})

ddo = ddo[
    ddo.full_text.str.len() >= 40
].copy()

ddo = ddo[
    ~ddo.full_text.str.lower().isin(
        ["[deleted]", "[removed]"]
    )
].copy()

ddo = ddo.drop_duplicates(
    "post_id"
).reset_index(drop=True)

assert len(ddo) == 948
assert ddo.post_id.is_unique
assert ddo.full_text.ne("").all()

# Verify compatibility with the 36 historical DDO candidates.
historical = folder / "DDO_historical_36_candidate_ids_PRIVATE.csv"

if historical.exists():
    old = pd.read_csv(historical, dtype=str)
    assert len(old) == 36
    assert set(old.post_id).issubset(set(ddo.post_id))
    print("Historical DDO IDs: 36/36 verified")

ddo.to_csv(destination, index=False)

print("\nDDO dataset ready:", len(ddo))
print("Saved:", destination)
print("CELL PASS")


In [ ]:

# CELL 23 — Restore complete external corpora from the 197-project archives.
# No old model predictions or human labels are imported.

from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

RUN_EXTERNAL = True
RUN_WOW = True

assert len(FT_TASKS) == 8
assert len(SEEDS) == 3
assert N_FOLDS == 5
assert all(
    ft_fold_path(t, r, s, k).exists()
    for t in FT_TASKS
    for r in FT_REPS
    for s in SEEDS
    for k in range(1, N_FOLDS + 1)
), "One or more of the 240 OOF files are missing."

DDO_CSV = INPUT / "DDO_eligible_948_PRIVATE.csv"
WOW_CSV = INPUT / "WoW_eligible_PRIVATE.csv"

OLD_DDO_IDS = existing_input(
    "DDO_historical_36_candidate_ids_PRIVATE.csv"
)
OLD_WOW_IDS = existing_input(
    "WoW_historical_5243_candidate_ids_PRIVATE.csv"
)

OLD_DDO_ROOT = DRIVE / "MMORPG_DDO_External_Application"

OLD_WOW_ROOTS = [
    DRIVE
    / "MMORPG_WOW_External_Application-20260920T135235Z-1-001"
    / "MMORPG_WOW_External_Application",
    DRIVE
    / "MMORPG_WOW_External_Application"
    / "MMORPG_WOW_External_Application",
    DRIVE / "MMORPG_WOW_External_Application",
]


def choose_source(paths, label):
    hits = list(dict.fromkeys(
        p.resolve() for p in paths if p.is_file()
    ))
    if not hits:
        return None
    if len(hits) > 1:
        digests = {sha256_file(p) for p in hits}
        assert len(digests) == 1, (
            f"{label}: conflicting copies. Inspect these paths:\n"
            + "\n".join(map(str, hits))
        )
    return hits[0]


def normalize_external(frame, name):
    z = frame.copy()

    for alias in ("ddo_post_id", "wow_post_id"):
        if "post_id" not in z.columns and alias in z.columns:
            z = z.rename(columns={alias: "post_id"})
        elif "post_id" in z.columns and alias in z.columns:
            assert (
                z.post_id.fillna("").astype(str).str.strip().equals(
                    z[alias].fillna("").astype(str).str.strip()
                )
            ), f"{name}: inconsistent ID columns."

    assert "post_id" in z.columns, f"{name}: missing post_id"

    z["post_id"] = z.post_id.fillna("").astype(str).str.strip()

    if "full_text" not in z.columns:
        title = (
            "title" if "title" in z.columns
            else "title_std" if "title_std" in z.columns
            else None
        )
        body = (
            "selftext" if "selftext" in z.columns
            else "body_std" if "body_std" in z.columns
            else None
        )
        assert title and body, f"{name}: missing text columns"

        z["full_text"] = (
            z[title].fillna("").astype(str).str.strip()
            + " "
            + z[body].fillna("").astype(str).str.strip()
        ).str.strip()

    z["full_text"] = (
        z.full_text.fillna("").astype(str).str.strip()
    )

    expected_n = {"DDO": 948, "WoW": 265009}[name]

    assert len(z) == expected_n, (
        f"{name}: expected {expected_n:,}, got {len(z):,}"
    )
    assert z.post_id.ne("").all()
    assert z.post_id.is_unique
    assert z.full_text.ne("").all()
    assert z.full_text.str.len().ge(40).all(), (
        f"{name}: source eligibility has changed"
    )
    assert not set(z.post_id).intersection(set(labels.post_id))

    print(
        name,
        "eligible rows:", len(z),
        "| unique IDs:", z.post_id.nunique()
    )

    if "subreddit" in z.columns:
        print(
            name,
            "source subreddit values:",
            z.subreddit.value_counts().head(5).to_dict()
        )

    return z[["post_id", "full_text"]].copy()


# Restore DDO from the previously saved canonical CSV.
DDO_SOURCE = (
    DDO_CSV if DDO_CSV.is_file()
    else choose_source([
        OLD_DDO_ROOT
        / "02_predictions"
        / "DDO_canonical_corpus.csv"
    ], "DDO")
)

if DDO_SOURCE is None:
    raise FileNotFoundError(
        "DDO canonical CSV not found. Restore:\n"
        "MMORPG_DDO_External_Application/"
        "02_predictions/DDO_canonical_corpus.csv"
    )

DDO = normalize_external(
    pd.read_csv(
        DDO_SOURCE,
        dtype=str,
        keep_default_na=False,
        low_memory=False
    ),
    "DDO"
)

if DDO_SOURCE != DDO_CSV:
    atomic_csv(DDO, DDO_CSV)


# Restore WoW from the earlier canonical Parquet.
wow_candidates = []

for root in OLD_WOW_ROOTS:
    if root.is_dir():
        wow_candidates.extend(
            sorted(root.rglob("WOW_canonical_corpus.parquet"))
        )

WOW_SOURCE = (
    WOW_CSV if WOW_CSV.is_file()
    else choose_source(wow_candidates, "WoW")
)

if WOW_SOURCE is None:
    raise FileNotFoundError(
        "WoW canonical Parquet not found. Search the old "
        "WoW project for WOW_canonical_corpus.parquet"
    )

if WOW_SOURCE.suffix.lower() == ".parquet":
    WOW = normalize_external(
        pd.read_parquet(WOW_SOURCE), "WoW"
    )
else:
    WOW = normalize_external(
        pd.read_csv(
            WOW_SOURCE,
            dtype=str,
            keep_default_na=False,
            low_memory=False
        ),
        "WoW"
    )

if WOW_SOURCE != WOW_CSV:
    atomic_csv(WOW, WOW_CSV)


# Freeze normalized external input sources.
EXTERNAL_DATA = {"DDO": DDO, "WoW": WOW}
EXTERNAL_PATHS = {"DDO": DDO_CSV, "WoW": WOW_CSV}

EXTERNAL_SHA = {
    name: sha256_file(path)
    for name, path in EXTERNAL_PATHS.items()
}

EXTERNAL_MANIFEST = {
    "protocol": "v6_58_18_original_canonical_external_sources",
    "model": MODEL_NAME,
    "revision": MODEL_REVISION,
    "training_label_sha256": INPUT_SHA,
    "master_sha256": MASTER_SHA,
    "model_cohorts": ["58", "18v2"],
    "thresholds": [0.60, 0.70],
    "min_votes": 3,
    "model_families": 4,
    "no_external_label_training": True,
    "external_sources": {
        name: {
            "source_path": str(
                DDO_SOURCE if name == "DDO" else WOW_SOURCE
            ),
            "source_sha256": sha256_file(
                DDO_SOURCE if name == "DDO" else WOW_SOURCE
            ),
            "normalized_path": str(EXTERNAL_PATHS[name]),
            "normalized_sha256": EXTERNAL_SHA[name],
            "n": len(EXTERNAL_DATA[name])
        }
        for name in EXTERNAL_DATA
    }
}

manifest_path = EXTERNAL / "external_run_manifest.json"

if manifest_path.exists():
    previous = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )
    for key in (
        "model",
        "revision",
        "training_label_sha256",
        "master_sha256"
    ):
        assert previous[key] == EXTERNAL_MANIFEST[key], (
            f"External cache conflict: {key}"
        )

    for name, info in EXTERNAL_MANIFEST["external_sources"].items():
        if name in previous.get("external_sources", {}):
            old = previous["external_sources"][name]
            old_hash = old.get(
                "normalized_sha256", old.get("sha256")
            )
            assert old_hash == info["normalized_sha256"], (
                f"{name}: stale external cache. Use a new run folder."
            )

save_json(manifest_path, EXTERNAL_MANIFEST)

print("CELL 23 PASS")
print("DDO posts:", len(DDO))
print("WoW posts:", len(WOW))
print("Normalized datasets:", INPUT)
print(
    "Historical ID files available:",
    OLD_DDO_IDS.exists(),
    OLD_WOW_IDS.exists()
)
print(
    "Check the original WoW subreddit value against "
    "the subreddit name reported in the manuscript."
)


In [ ]:

# ============================================================
# CELL 24 — FINAL CORRECTED EXTERNAL APPLICATION
# ============================================================
#
# Primary model: 58 positive / 116 negative MMORPG posts
# Strict model: 18 positive / 36 negative MMORPG posts
#
# External corpora:
#   DDO: 948 eligible posts
#   WoW: 265,009 eligible posts
#
# Models:
#   - MentalBERT embeddings + Logistic Regression
#   - MentalBERT embeddings + Linear SVM
#   - MentalBERT embeddings + Random Forest
#   - Fine-tuned MentalBERT (seeds 42, 123, 2026)
#
# Representations:
#   - raw
#   - broad_masked
#
# The existing 240 grouped-CV fold results are preserved.
# No external human labels are used for model training.
#
# Requires Cell 23 to have completed in this runtime.
# Requires GPU and sufficient Google Drive storage.
# ============================================================

from pathlib import Path
import hashlib
import json
import os
import gc

import numpy as np
import pandas as pd
import torch
import joblib

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling
from numpy.lib.format import open_memmap

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from transformers import AutoModelForSequenceClassification


# ============================================================
# 1. PRE-FLIGHT VALIDATION
# ============================================================

assert RUN_EXTERNAL and RUN_WOW, (
    "Cell 23 must enable both external corpora."
)

assert set(EXTERNAL_DATA) == {"DDO", "WoW"}

assert len(EXTERNAL_DATA["DDO"]) == 948
assert len(EXTERNAL_DATA["WoW"]) == 265009

assert torch.cuda.is_available(), (
    "Enable a GPU runtime before executing Cell 24."
)

assert list(FT_REPS) == ["raw", "broad_masked"]
assert list(SEEDS) == [42, 123, 2026]
assert MAX_TOKENS == 512

assert FT_TRAINING_PROTOCOL == (
    "r1_constant_lr_no_scheduler_no_clip_amp"
)

# Verify existing grouped-CV predictions.

expected_cv = (
    len(FT_TASKS)
    * len(FT_REPS)
    * len(SEEDS)
    * N_FOLDS
)

assert expected_cv == 240

for task in FT_TASKS:
    for rep in FT_REPS:
        for seed in SEEDS:
            for fold in range(1, N_FOLDS + 1):
                path = ft_fold_path(
                    task, rep, seed, fold
                )

                assert path.is_file(), (
                    f"Missing grouped-CV result: {path}"
                )

print(
    "PRECHECK PASS:",
    expected_cv,
    "existing grouped-CV prediction files found."
)


# ============================================================
# 2. WOW SOURCE PROVENANCE AUDIT
# ============================================================

old_wow_parquet = (
    INPUT
    / "WOW_197_main_strict_three_seed_predictions.parquet"
)

if old_wow_parquet.is_file():

    import pyarrow.parquet as pq

    columns = pq.ParquetFile(
        old_wow_parquet
    ).schema.names

    if "subreddit" in columns:

        subreddit = pd.read_parquet(
            old_wow_parquet,
            columns=["subreddit"]
        )["subreddit"]

        counts = (
            subreddit.astype(str)
            .str.lower()
            .value_counts(dropna=False)
            .to_dict()
        )

        print(
            "ARCHIVED WOW SUBREDDIT VALUES:",
            counts
        )

        if set(counts) == {"wow"}:
            print(
                "MANUSCRIPT CHECK: "
                "Original corpus corresponds to r/wow."
            )

    else:
        print(
            "SUBREDDIT AUDIT: source column absent "
            "from archived parquet. "
            "Verify original metadata before manuscript submission."
        )

else:
    print(
        "SUBREDDIT AUDIT: archived source "
        "not available for metadata verification."
    )


# ============================================================
# 3. HISTORICAL ID LINKAGE
# ============================================================

for corpus, old_path, expected in (
    ("DDO", OLD_DDO_IDS, 36),
    ("WoW", OLD_WOW_IDS, 5243)
):

    if not old_path.is_file():
        print(
            "OPTIONAL HISTORICAL IDS MISSING:",
            corpus
        )
        continue

    old = pd.read_csv(
        old_path,
        dtype=str,
        keep_default_na=False
    )

    assert "post_id" in old.columns
    assert len(old) == expected
    assert old.post_id.nunique() == expected

    external_ids = set(
        EXTERNAL_DATA[corpus].post_id
    )

    matched = int(
        old.post_id.isin(external_ids).sum()
    )

    assert matched == expected, (
        f"{corpus}: historical ID mismatch "
        f"{matched}/{expected}"
    )

    print(
        f"{corpus} historical ID linkage PASS:",
        f"{matched}/{expected}"
    )


# ============================================================
# 4. FINAL TRAINING DATA
# ============================================================

FINAL_TRAIN = {
    "58": DATASETS["rq2_primary_58v116"],
    "18v2": DATASETS["rq2_strict18v36"]
}

assert (
    len(FINAL_TRAIN["58"]),
    int(FINAL_TRAIN["58"].label.sum())
) == (174, 58)

assert (
    len(FINAL_TRAIN["18v2"]),
    int(FINAL_TRAIN["18v2"].label.sum())
) == (54, 18)

strict_negative_ids = set(
    FINAL_TRAIN["18v2"]
    .loc[lambda x: x.label.eq(0), "post_id"]
)

primary_negative_ids = set(
    FINAL_TRAIN["58"]
    .loc[lambda x: x.label.eq(0), "post_id"]
)

assert strict_negative_ids.issubset(
    primary_negative_ids
)

print("FINAL TRAINING DATA VERIFIED")
print("Primary:", len(FINAL_TRAIN["58"]))
print("Strict:", len(FINAL_TRAIN["18v2"]))


# ============================================================
# 5. FROZEN EXTERNAL RUN CONFIGURATION
# ============================================================

EXTERNAL_PROTOCOL_VERSION = (
    "external_v6_58_18v2_repro_20261113_03"
)

full_training_sha = {
    "58": sha256_file(
        DATA / "rq2_primary_58v116_PRIVATE.csv"
    ),
    "18v2": sha256_file(
        DATA / "rq2_strict18v36_PRIVATE.csv"
    )
}

EXTERNAL_RUN_CONFIG = {
    "protocol_version": EXTERNAL_PROTOCOL_VERSION,
    "training_set_sha256": full_training_sha,
    "label_sha256": INPUT_SHA,
    "master_sha256": MASTER_SHA,
    "source_sha256": EXTERNAL_SHA,
    "model": MODEL_NAME,
    "revision": MODEL_REVISION,
    "max_tokens": MAX_TOKENS,
    "mask_regex": BROAD.pattern,
    "seeds": SEEDS,
    "fine_tuning_epochs": MAX_FT_EPOCHS,
    "fine_tuning_batch": FT_BATCH,
    "fine_tuning_accumulation": FT_ACCUM,
    "fine_tuning_lr": FT_LR,
    "training_protocol": FT_TRAINING_PROTOCOL,
    "embedding": (
        "SentenceTransformer mean pooling, "
        "unnormalised, 768 dimensions"
    ),
    "classical_transform": (
        "PCA50 -> StandardScaler -> classifier"
    ),
    "classical_families": [
        "logistic",
        "linear_svm",
        "random_forest"
    ],
    "finetuned_family": "MentalBERT three-seed mean",
    "external_human_labels_used": False
}

config_string = json.dumps(
    EXTERNAL_RUN_CONFIG,
    sort_keys=True
)

run_key = hashlib.sha256(
    config_string.encode()
).hexdigest()[:14]

EXTERNAL_RUN = (
    EXTERNAL
    / f"{EXTERNAL_PROTOCOL_VERSION}_{run_key}"
)

EXTERNAL_RUN.mkdir(
    parents=True,
    exist_ok=True
)

config_path = EXTERNAL_RUN / "run_config.json"

if config_path.is_file():

    saved_config = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    assert saved_config == EXTERNAL_RUN_CONFIG, (
        "External run configuration mismatch."
    )

else:
    save_json(
        config_path,
        EXTERNAL_RUN_CONFIG
    )

print(
    "Frozen external output folder:",
    EXTERNAL_RUN
)


# ============================================================
# 6. LOAD PRETRAINED MENTALBERT
# ============================================================

encoder = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device="cuda"
)

encoder.max_seq_length = MAX_TOKENS


# ============================================================
# 7. CORRECTED POOLING AUDIT
# ============================================================
#
# sentence-transformers versions can expose pooling as:
#
# New API:
#   {"pooling_mode": "mean"}
#
# Legacy API:
#   {"pooling_mode_mean_tokens": True, ...}
#
# Do not rely on missing attributes of the Pooling module.
# ============================================================

poolers = [
    module
    for module in encoder._modules.values()
    if isinstance(module, Pooling)
]

assert len(poolers) == 1, (
    f"Expected one pooling module, got {len(poolers)}"
)

pooler = poolers[0]
pool_config = pooler.get_config_dict()

assert isinstance(pool_config, dict), (
    f"Unexpected pooling configuration: {pool_config!r}"
)

if "pooling_mode" in pool_config:

    modes = pool_config["pooling_mode"]

    if isinstance(modes, str):
        modes = [modes]

    assert set(modes) == {"mean"}, (
        f"Expected mean pooling only: {pool_config}"
    )

else:

    assert bool(
        pool_config.get(
            "pooling_mode_mean_tokens",
            False
        )
    ), (
        f"Mean pooling not enabled: {pool_config}"
    )

    for field in (
        "pooling_mode_cls_token",
        "pooling_mode_max_tokens",
        "pooling_mode_mean_sqrt_len_tokens",
        "pooling_mode_weightedmean_tokens",
        "pooling_mode_lasttoken"
    ):

        assert not bool(
            pool_config.get(field, False)
        ), (
            f"Unexpected pooling enabled: {field}"
        )

assert (
    encoder.get_sentence_embedding_dimension()
    == 768
)

print("MENTALBERT POOLING AUDIT: PASS")
print("Pooling configuration:", pool_config)
print(
    "Embedding dimension:",
    encoder.get_sentence_embedding_dimension()
)


# ============================================================
# 8. RESUMABLE EXTERNAL EMBEDDINGS
# ============================================================

def external_embeddings(
    name,
    representation,
    source,
    chunk_size=1024
):

    folder = (
        EXTERNAL_RUN
        / "embedding_cache"
        / name
        / representation
    )

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    ids_hash = hashlib.sha256(
        "\n".join(
            source.post_id.astype(str)
        ).encode()
    ).hexdigest()

    signature = {
        "source_sha256": EXTERNAL_SHA[name],
        "ids_sha256": ids_hash,
        "representation": representation,
        "model": MODEL_NAME,
        "revision": MODEL_REVISION,
        "mask_regex": (
            BROAD.pattern
            if representation == "broad_masked"
            else None
        ),
        "max_tokens": MAX_TOKENS,
        "rows": len(source),
        "dimension": 768,
        "chunk_size": chunk_size
    }

    manifest = folder / "signature.json"

    if manifest.is_file():

        assert json.loads(
            manifest.read_text(encoding="utf-8")
        ) == signature, (
            "External embedding cache mismatch."
        )

    else:

        assert not list(
            folder.glob("chunk_*.npy")
        ), (
            "Unverified embedding chunks found."
        )

        save_json(manifest, signature)

    ranges = [
        (
            start,
            min(start + chunk_size, len(source))
        )
        for start in range(
            0,
            len(source),
            chunk_size
        )
    ]

    for start, end in tqdm(
        ranges,
        desc=f"{name} pretrained {representation}"
    ):

        path = (
            folder
            / f"chunk_{start:07d}_{end:07d}.npy"
        )

        if path.is_file():

            existing = np.load(
                path,
                mmap_mode="r"
            )

            assert existing.shape == (
                end - start,
                768
            )

            assert existing.dtype == np.float32
            del existing
            continue

        texts = (
            source.full_text
            .iloc[start:end]
            .astype(str)
            .tolist()
        )

        if representation == "broad_masked":

            texts = [
                mask_text(text)
                for text in texts
            ]

        values = np.asarray(
            encoder.encode(
                texts,
                batch_size=16,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            ),
            dtype=np.float32
        )

        assert values.shape == (
            end - start,
            768
        )

        assert np.isfinite(values).all()

        temporary = path.with_name(
            path.stem + ".partial.npy"
        )

        np.save(
            temporary,
            values
        )

        os.replace(
            temporary,
            path
        )

    combined = (
        folder
        / f"all_{len(source)}_float32.npy"
    )

    if not combined.is_file():

        temporary = (
            folder
            / f"all_{len(source)}_float32.partial.npy"
        )

        matrix = open_memmap(
            temporary,
            mode="w+",
            dtype="float32",
            shape=(
                len(source),
                768
            )
        )

        for start, end in tqdm(
            ranges,
            desc=f"Assemble {name} {representation}"
        ):

            chunk = np.load(
                folder
                / f"chunk_{start:07d}_{end:07d}.npy",
                mmap_mode="r"
            )

            matrix[start:end] = chunk

        matrix.flush()
        del matrix

        os.replace(
            temporary,
            combined
        )

    result = np.load(
        combined,
        mmap_mode="r"
    )

    assert result.shape == (
        len(source),
        768
    )

    assert result.dtype == np.float32

    return result


# ============================================================
# 9. EXTERNAL EMBEDDING MATRICES
# ============================================================

EXTERNAL_EMBED = {}

for corpus_name, frame in EXTERNAL_DATA.items():

    for representation in FT_REPS:

        EXTERNAL_EMBED[
            (corpus_name, representation)
        ] = external_embeddings(
            corpus_name,
            representation,
            frame
        )

print(
    "PASS: external pretrained embeddings cached "
    "and source-bound."
)


# ============================================================
# 10. CLASSICAL MODEL TRAINING AND SCORING
# ============================================================
#
# Important:
# PCA must be fitted BEFORE StandardScaler, matching the
# original external v6 analysis.
#
# All transformations are fitted exclusively on MMORPG.
# ============================================================

for cohort, train in FINAL_TRAIN.items():

    for representation in FT_REPS:

        model_dir = (
            EXTERNAL_RUN
            / "frozen_models"
            / cohort
        )

        model_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        classical_path = (
            model_dir
            / f"classical_{representation}.joblib"
        )

        training_texts = (
            train.text.astype(str).tolist()
        )

        if representation == "broad_masked":

            training_texts = [
                mask_text(text)
                for text in training_texts
            ]

        if classical_path.is_file():

            saved = joblib.load(
                classical_path
            )

            assert (
                saved["source_sha256"]
                == full_training_sha[cohort]
            )

            assert (
                saved["representation"]
                == representation
            )

            assert saved["config_key"] == run_key

            pca = saved["pca"]
            scaler = saved["scaler"]
            families = saved["families"]

            print(
                "CACHED classical model:",
                cohort,
                representation
            )

        else:

            train_vectors = np.asarray(
                encoder.encode(
                    training_texts,
                    batch_size=16,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=False
                ),
                dtype=np.float32
            )

            y = train.label.to_numpy(int)

            assert len(train_vectors) == len(y)
            assert len(set(y)) == 2

            # PCA first.
            pca = PCA(
                n_components=50,
                svd_solver="randomized",
                random_state=42
            )

            transformed = pca.fit_transform(
                train_vectors
            )

            # StandardScaler second.
            scaler = StandardScaler()

            transformed = scaler.fit_transform(
                transformed
            )

            families = {

                "logistic": LogisticRegression(
                    C=1.0,
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=3000,
                    random_state=42
                ),

                "linear_svm": SVC(
                    kernel="linear",
                    C=1.0,
                    probability=True,
                    class_weight="balanced",
                    random_state=42
                ),

                "random_forest": RandomForestClassifier(
                    n_estimators=400,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=42
                )
            }

            for estimator in families.values():
                estimator.fit(
                    transformed,
                    y
                )

            joblib.dump(
                {
                    "source_sha256": full_training_sha[cohort],
                    "representation": representation,
                    "config_key": run_key,
                    "pca": pca,
                    "scaler": scaler,
                    "families": families
                },
                classical_path
            )

            print(
                "SAVED classical model:",
                cohort,
                representation
            )

        # Score both external corpora using the frozen models.

        for corpus_name, frame in EXTERNAL_DATA.items():

            destination = (
                EXTERNAL_RUN
                / f"{corpus_name}_{cohort}_{representation}_three_classical_PRIVATE.csv"
            )

            if destination.is_file():

                cached = pd.read_csv(
                    destination,
                    dtype={"post_id": str}
                )

                assert (
                    cached.post_id.tolist()
                    == frame.post_id.tolist()
                )

                assert cached[
                    [
                        "logistic",
                        "linear_svm",
                        "random_forest"
                    ]
                ].apply(np.isfinite).all().all()

                print(
                    "CACHED classical scores:",
                    destination.name
                )

                continue

            external_vectors = EXTERNAL_EMBED[
                (corpus_name, representation)
            ]

            output_blocks = []

            for start in tqdm(
                range(0, len(frame), 2048),
                desc=(
                    f"{corpus_name} "
                    f"{cohort} {representation} classical"
                )
            ):

                end = min(
                    start + 2048,
                    len(frame)
                )

                batch_vectors = np.asarray(
                    external_vectors[start:end]
                )

                # Same order used during classical training.
                transformed = scaler.transform(
                    pca.transform(batch_vectors)
                )

                block = pd.DataFrame({
                    "post_id": (
                        frame.post_id
                        .iloc[start:end]
                        .tolist()
                    )
                })

                for model_name, estimator in families.items():

                    block[model_name] = (
                        estimator.predict_proba(
                            transformed
                        )[:, 1]
                    )

                output_blocks.append(block)

            scores = pd.concat(
                output_blocks,
                ignore_index=True
            )

            assert (
                scores.post_id.tolist()
                == frame.post_id.tolist()
            )

            atomic_csv(
                scores,
                destination
            )

            print(
                "SAVED classical scores:",
                destination.name
            )


# ============================================================
# 11. RELEASE PRETRAINED ENCODER
# ============================================================
#
# Release only after all classical training/scoring
# operations are complete.
# ============================================================

del encoder
del EXTERNAL_EMBED

gc.collect()
torch.cuda.empty_cache()

print("CLASSICAL EXTERNAL SCORING COMPLETE")


# ============================================================
# 12. RESUMABLE FINE-TUNED MODEL SCORING
# ============================================================

def external_finetuned_score(
    model,
    corpus,
    frame,
    cohort,
    representation,
    seed,
    model_hash,
    chunk_size=512
):

    destination = (
        EXTERNAL_RUN
        / f"{corpus}_{cohort}_{representation}_MentalBERT_seed{seed}_PRIVATE.csv"
    )

    folder = (
        EXTERNAL_RUN
        / "finetuned_score_chunks"
        / corpus
        / cohort
        / representation
        / f"seed{seed}"
    )

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    ids_hash = hashlib.sha256(
        "\n".join(frame.post_id).encode()
    ).hexdigest()

    signature = {
        "dataset_sha256": EXTERNAL_SHA[corpus],
        "ordered_ids_sha256": ids_hash,
        "model_weights_sha256": model_hash,
        "cohort": cohort,
        "representation": representation,
        "seed": seed,
        "chunk_size": chunk_size,
        "max_tokens": MAX_TOKENS
    }

    signature_path = folder / "signature.json"

    if signature_path.is_file():

        assert json.loads(
            signature_path.read_text(
                encoding="utf-8"
            )
        ) == signature, (
            "Stale external score chunks detected."
        )

    else:

        assert not list(
            folder.glob("*.csv")
        ), (
            "Unverified external score chunks detected."
        )

        save_json(
            signature_path,
            signature
        )

    if destination.is_file():

        cached = pd.read_csv(
            destination,
            dtype={"post_id": str}
        )

        assert (
            cached.post_id.tolist()
            == frame.post_id.tolist()
        )

        assert cached.score.between(0, 1).all()
        assert np.isfinite(cached.score).all()

        print(
            "CACHED fine-tuned scores:",
            destination.name
        )

        return

    chunks = []

    for start in tqdm(
        range(0, len(frame), chunk_size),
        desc=(
            f"{corpus} {cohort} "
            f"{representation} seed{seed}"
        )
    ):

        end = min(
            start + chunk_size,
            len(frame)
        )

        chunk_path = (
            folder
            / f"{start:07d}_{end:07d}.csv"
        )

        expected_ids = (
            frame.post_id
            .iloc[start:end]
            .tolist()
        )

        if chunk_path.is_file():

            block = pd.read_csv(
                chunk_path,
                dtype={"post_id": str}
            )

            assert (
                block.post_id.tolist()
                == expected_ids
            )

            assert block.score.between(0, 1).all()
            assert np.isfinite(block.score).all()

        else:

            texts = (
                frame.full_text
                .iloc[start:end]
                .tolist()
            )

            if representation == "broad_masked":

                texts = [
                    mask_text(text)
                    for text in texts
                ]

            probabilities = mentalbert_predict(
                model,
                texts
            )

            assert len(probabilities) == len(texts)
            assert np.isfinite(probabilities).all()

            block = pd.DataFrame({
                "post_id": expected_ids,
                "score": probabilities
            })

            atomic_csv(
                block,
                chunk_path
            )

        chunks.append(block)

    result = pd.concat(
        chunks,
        ignore_index=True
    )

    assert (
        result.post_id.tolist()
        == frame.post_id.tolist()
    )

    atomic_csv(
        result,
        destination
    )

    print(
        "SAVED fine-tuned scores:",
        destination.name
    )


# ============================================================
# 13. FINAL FULL-TRAINING MENTALBERT MODELS
# ============================================================
#
# 2 training cohorts x 2 representations x 3 seeds
# = 12 final models.
#
# These are independent of the 240 grouped-CV runs.
# ============================================================

for cohort, train in FINAL_TRAIN.items():

    train_labels = train.label.to_numpy(int)

    for representation in FT_REPS:

        training_texts = (
            train.text.astype(str).tolist()
        )

        if representation == "broad_masked":

            training_texts = [
                mask_text(text)
                for text in training_texts
            ]

        for seed in SEEDS:

            model_dir = (
                EXTERNAL_RUN
                / "frozen_models"
                / cohort
                / f"MentalBERT_{representation}_seed{seed}"
            )

            marker_path = (
                model_dir
                / "TRAINING_COMPLETE.json"
            )

            expected_marker = {
                "source_sha256": full_training_sha[cohort],
                "label_sha256": INPUT_SHA,
                "model_revision": MODEL_REVISION,
                "seed": seed,
                "cohort": cohort,
                "representation": representation,
                "training_protocol": FT_TRAINING_PROTOCOL,
                "run_key": run_key
            }

            if marker_path.is_file():

                existing_marker = json.loads(
                    marker_path.read_text(
                        encoding="utf-8"
                    )
                )

                assert (
                    existing_marker == expected_marker
                ), (
                    "Fine-tuned checkpoint provenance mismatch."
                )

                assert (
                    model_dir / "config.json"
                ).is_file()

                print(
                    "CACHED final MentalBERT:",
                    cohort,
                    representation,
                    seed
                )

            else:

                model = fit_mentalbert(
                    training_texts,
                    train_labels,
                    seed,
                    save_model_dir=model_dir
                )

                del model

                gc.collect()
                torch.cuda.empty_cache()

                save_json(
                    marker_path,
                    expected_marker
                )

                print(
                    "SAVED final MentalBERT:",
                    cohort,
                    representation,
                    seed
                )

            weights_path = (
                model_dir
                / "model.safetensors"
            )

            assert weights_path.is_file(), (
                f"Missing saved weights: {weights_path}"
            )

            model_hash = sha256_file(
                weights_path
            )

            model = (
                AutoModelForSequenceClassification
                .from_pretrained(model_dir)
                .to("cuda")
            )

            for corpus_name, frame in EXTERNAL_DATA.items():

                external_finetuned_score(
                    model=model,
                    corpus=corpus_name,
                    frame=frame,
                    cohort=cohort,
                    representation=representation,
                    seed=seed,
                    model_hash=model_hash
                )

            del model

            gc.collect()
            torch.cuda.empty_cache()


# ============================================================
# 14. FINAL FILE COMPLETENESS AUDIT
# ============================================================

EXPECTED_FILES_24 = []

for corpus in EXTERNAL_DATA:

    for cohort in FINAL_TRAIN:

        for representation in FT_REPS:

            EXPECTED_FILES_24.append(
                EXTERNAL_RUN
                / f"{corpus}_{cohort}_{representation}_three_classical_PRIVATE.csv"
            )

            for seed in SEEDS:

                EXPECTED_FILES_24.append(
                    EXTERNAL_RUN
                    / f"{corpus}_{cohort}_{representation}_MentalBERT_seed{seed}_PRIVATE.csv"
                )

assert len(EXPECTED_FILES_24) == 32

missing_files = [
    path
    for path in EXPECTED_FILES_24
    if not path.is_file()
]

assert not missing_files, (
    "External scoring files missing:\n"
    + "\n".join(map(str, missing_files))
)

print()
print("=" * 65)
print("CELL 24 PASS")
print("=" * 65)
print("DDO posts:", len(EXTERNAL_DATA["DDO"]))
print("WoW posts:", len(EXTERNAL_DATA["WoW"]))
print("Final MentalBERT models:", 12)
print("External score files:", len(EXPECTED_FILES_24))
print("Previous grouped-CV jobs preserved:", expected_cv)
print("Output directory:", EXTERNAL_RUN)
print("=" * 65)


In [ ]:

# ============================================================
# CELL 25 — EXTERNAL CONSENSUS AND BLINDED HUMAN REVIEW
# ============================================================
# Uses the new frozen Cell 24 outputs.
# Historical IDs are used for comparison only.
# No historical predictions or human labels enter training.
# Existing manual-review files are never overwritten.

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

assert RUN_EXTERNAL and RUN_WOW
assert "EXTERNAL_RUN" in globals()
assert EXTERNAL_RUN.is_dir()

assert "EXPECTED_FILES_24" in globals()
assert len(EXPECTED_FILES_24) == 32
assert all(p.is_file() for p in EXPECTED_FILES_24)

assert set(EXTERNAL_DATA) == {"DDO", "WoW"}
assert len(EXTERNAL_DATA["DDO"]) == 948
assert len(EXTERNAL_DATA["WoW"]) == 265009

assert (
    EXTERNAL_RUN / "run_config.json"
).is_file()

print("CELL 25 — EXTERNAL CONSENSUS")
print("Using frozen run:", EXTERNAL_RUN)


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def save_blind_once(frame, path):
    """
    Create a blinded review sheet without overwriting
    any previously completed or partially completed reviews.
    """
    assert frame.post_id.is_unique

    if path.exists():
        old = pd.read_csv(
            path,
            dtype={"post_id": str},
            keep_default_na=False
        )

        assert old.post_id.tolist() == frame.post_id.tolist(), (
            f"Existing review sheet differs: {path}. "
            "Preserve existing human ratings and audit first."
        )

        print("PRESERVED review sheet:", path.name)

    else:
        atomic_csv(frame, path)
        print("SAVED review sheet:", path.name)


def read_historical_ids(path, expected):
    """
    Read old candidate IDs for a descriptive overlap audit.
    Do not interpret historical selection as a gold standard.
    """
    if not path.is_file():
        print("Historical reference unavailable:", path.name)
        return None

    z = pd.read_csv(
        path,
        dtype={"post_id": str},
        keep_default_na=False
    )

    assert "post_id" in z.columns
    assert len(z) == expected

    ids = set(z.post_id.astype(str).str.strip())

    assert len(ids) == expected
    assert "" not in ids

    return ids


def make_blind_review(frame, three_reviewers=True):
    """
    Reviewer sheet containing text but no model probabilities,
    membership flags, or selection strata.
    """
    out = frame[["post_id", "full_text"]].copy()

    fields = ["Reviewer_1", "Reviewer_2"]

    if three_reviewers:
        fields.append("Reviewer_3")

    fields.extend([
        "Final_manual_decision",
        "Manual_evidence_notes"
    ])

    for col in fields:
        out[col] = ""

    return out


# ============================================================
# 2. COMPUTE FOUR-FAMILY CONSENSUS
# ============================================================

selection_rows = []
MEMBERSHIP = {}

for corpus, source in EXTERNAL_DATA.items():

    membership = pd.DataFrame({
        "post_id": source.post_id.astype(str).tolist()
    })

    assert membership.post_id.is_unique

    for cohort in ("58", "18v2"):

        for representation in ("raw", "broad_masked"):

            prefix = (
                EXTERNAL_RUN
                / f"{corpus}_{cohort}_{representation}"
            )

            classical_path = Path(
                str(prefix)
                + "_three_classical_PRIVATE.csv"
            )

            classic = pd.read_csv(
                classical_path,
                dtype={"post_id": str}
            )

            assert (
                classic.post_id.tolist()
                == membership.post_id.tolist()
            )

            # Load all three independently fitted MentalBERT seeds.
            seed_probabilities = []

            for seed in SEEDS:

                path = (
                    EXTERNAL_RUN
                    / (
                        f"{corpus}_{cohort}_{representation}"
                        f"_MentalBERT_seed{seed}_PRIVATE.csv"
                    )
                )

                z = pd.read_csv(
                    path,
                    dtype={"post_id": str}
                )

                assert (
                    z.post_id.tolist()
                    == membership.post_id.tolist()
                )

                probabilities = z.score.to_numpy(float)

                assert np.isfinite(probabilities).all()
                assert (
                    (probabilities >= 0)
                    & (probabilities <= 1)
                ).all()

                seed_probabilities.append(probabilities)

            classic["finetuned_3seed_mean"] = np.mean(
                np.stack(seed_probabilities),
                axis=0
            )

            families = [
                "logistic",
                "linear_svm",
                "random_forest",
                "finetuned_3seed_mean"
            ]

            assert classic[families].apply(
                np.isfinite
            ).all().all()

            assert classic[families].ge(0).all().all()
            assert classic[families].le(1).all().all()

            # At least three of four model families must
            # exceed the specified threshold.
            for threshold in (0.60, 0.70):

                votes = (
                    classic[families] >= threshold
                ).sum(axis=1)

                th = int(round(100 * threshold))

                key = (
                    f"{cohort}_{representation}_{th}"
                )

                membership[
                    f"selected_{key}"
                ] = votes.ge(3).to_numpy(bool)

                classic[
                    f"votes_{th}"
                ] = votes

                selection_rows.append({
                    "corpus": corpus,
                    "cohort": cohort,
                    "representation": representation,
                    "threshold": threshold,
                    "eligible_posts": len(source),
                    "candidates": int(
                        membership[
                            f"selected_{key}"
                        ].sum()
                    ),
                    "minimum_votes": 3,
                    "model_families": 4
                })

            # Keep exact probabilities in a private,
            # non-blinded technical output.
            atomic_csv(
                classic,
                EXTERNAL_RUN
                / (
                    f"{corpus}_{cohort}_{representation}"
                    "_all_four_scores_PRIVATE.csv"
                )
            )


    # ========================================================
    # 3. COMBINE RAW / MASKED AND 58 / 18 SELECTIONS
    # ========================================================

    for cohort in ("58", "18v2"):

        for threshold in (60, 70):

            membership[
                f"{cohort}_anyrep_{threshold}"
            ] = membership[
                [
                    f"selected_{cohort}_raw_{threshold}",
                    f"selected_{cohort}_broad_masked_{threshold}"
                ]
            ].any(axis=1)

    for threshold in (60, 70):

        membership[
            f"union_anycohort_{threshold}"
        ] = membership[
            [
                f"58_anyrep_{threshold}",
                f"18v2_anyrep_{threshold}"
            ]
        ].any(axis=1)

    assert (
        membership["union_anycohort_70"]
        <= membership["union_anycohort_60"]
    ).all()

    membership_file = (
        EXTERNAL_RUN
        / (
            f"{corpus}_58_vs_18v2_"
            "selection_membership_PRIVATE.csv"
        )
    )

    atomic_csv(
        membership,
        membership_file
    )

    MEMBERSHIP[corpus] = membership

    selected = membership["union_anycohort_60"]

    print()
    print("CORPUS:", corpus)
    print("Eligible:", len(source))
    print(
        "Selected at >= 0.60:",
        int(selected.sum())
    )
    print(
        "Selected at >= 0.70:",
        int(membership["union_anycohort_70"].sum())
    )


    # ========================================================
    # 4. COMPLETE BLINDED CANDIDATE-REVIEW FILE
    # ========================================================

    selected_posts = source.loc[
        selected.to_numpy(),
        ["post_id", "full_text"]
    ].copy()

    review_file = (
        EXTERNAL_RUN
        / (
            f"{corpus}_new_union_candidates_"
            "BLIND_MANUAL_REVIEW_PRIVATE.csv"
        )
    )

    save_blind_once(
        make_blind_review(
            selected_posts,
            three_reviewers=True
        ),
        review_file
    )


    # ========================================================
    # 5. DDO: RANDOM SAMPLE OF UNSELECTED POSTS
    # ========================================================

    if corpus == "DDO":

        unselected = source.loc[
            ~selected.to_numpy(),
            ["post_id", "full_text"]
        ].copy()

        random_sample = unselected.sample(
            n=min(100, len(unselected)),
            random_state=42
        )

        save_blind_once(
            make_blind_review(
                random_sample,
                three_reviewers=False
            ),
            EXTERNAL_RUN
            / (
                "DDO_unselected_random_100_"
                "BLIND_REVIEW_PRIVATE.csv"
            )
        )


    # ========================================================
    # 6. HISTORICAL 197-MODEL OVERLAP
    # ========================================================

    historical_file = (
        OLD_DDO_IDS
        if corpus == "DDO"
        else OLD_WOW_IDS
    )

    historical_count = (
        36 if corpus == "DDO" else 5243
    )

    historical_ids = read_historical_ids(
        historical_file,
        historical_count
    )

    if historical_ids is not None:

        current_source_ids = set(
            membership.post_id
        )

        assert historical_ids.issubset(
            current_source_ids
        ), (
            f"{corpus}: old candidate IDs do not "
            "fully match the current external corpus."
        )

        new_ids = set(
            membership.loc[selected, "post_id"]
        )

        comparison = pd.DataFrame({
            "post_id": sorted(
                historical_ids | new_ids
            )
        })

        comparison["old_candidate"] = (
            comparison.post_id.isin(historical_ids)
        )

        comparison["new_candidate"] = (
            comparison.post_id.isin(new_ids)
        )

        comparison["status"] = np.select(
            [
                (
                    comparison.old_candidate
                    & comparison.new_candidate
                ),
                (
                    comparison.old_candidate
                    & ~comparison.new_candidate
                ),
                (
                    ~comparison.old_candidate
                    & comparison.new_candidate
                )
            ],
            [
                "retained",
                "no_longer_selected",
                "newly_selected"
            ],
            default="CHECK"
        )

        atomic_csv(
            comparison,
            EXTERNAL_RUN
            / (
                f"{corpus}_old_vs_new_"
                "ID_overlap_PRIVATE.csv"
            )
        )

        print(
            "Historical overlap:",
            comparison.status.value_counts().to_dict()
        )


    # ========================================================
    # 7. WOW: STRATIFIED BLIND SAMPLE OF UP TO 400
    # ========================================================

    if corpus == "WoW" and selected.any():

        pool = membership.loc[
            selected
        ].copy()

        # Stratify by model-cohort selection.
        pool["cohort_group"] = np.select(
            [
                (
                    pool["58_anyrep_60"]
                    & pool["18v2_anyrep_60"]
                ),
                pool["58_anyrep_60"]
            ],
            [
                "both",
                "58_only"
            ],
            default="18_only"
        )

        # Stratify by raw/masked agreement.
        raw_selected = (
            pool["selected_58_raw_60"]
            | pool["selected_18v2_raw_60"]
        )

        masked_selected = (
            pool["selected_58_broad_masked_60"]
            | pool["selected_18v2_broad_masked_60"]
        )

        pool["representation_group"] = np.select(
            [
                raw_selected & masked_selected,
                raw_selected
            ],
            [
                "both",
                "raw_only"
            ],
            default="masked_only"
        )

        pool["threshold_group"] = np.where(
            pool["union_anycohort_70"],
            "0.70+",
            "0.60-0.69"
        )

        pool["stratum"] = (
            pool["cohort_group"]
            + " | "
            + pool["threshold_group"]
            + " | "
            + pool["representation_group"]
        )

        buckets = {
            key: group.index.to_numpy()
            for key, group in pool.groupby(
                "stratum",
                sort=True
            )
        }

        rng = np.random.default_rng(42)

        target = min(
            400,
            len(pool)
        )

        selected_indices = []
        selected_per_stratum = Counter()

        # Initial minimum coverage of non-empty strata.
        for key, indices in buckets.items():

            k = min(
                3,
                len(indices),
                target - len(selected_indices)
            )

            if k > 0:

                chosen = rng.choice(
                    indices,
                    size=k,
                    replace=False
                ).tolist()

                selected_indices.extend(chosen)
                selected_per_stratum[key] += k

        # Fill remaining capacity from strata with
        # proportionally more unselected records.
        while len(selected_indices) < target:

            active = [
                key
                for key, indices in buckets.items()
                if (
                    selected_per_stratum[key]
                    < len(indices)
                )
            ]

            assert active

            key = max(
                active,
                key=lambda x: (
                    len(buckets[x])
                    - selected_per_stratum[x]
                ) / len(buckets[x])
            )

            unused = np.setdiff1d(
                buckets[key],
                np.asarray(selected_indices),
                assume_unique=True
            )

            index = int(
                rng.choice(unused)
            )

            selected_indices.append(index)
            selected_per_stratum[key] += 1

        sample_meta = pool.loc[
            selected_indices,
            ["post_id", "stratum"]
        ].copy()

        population_counts = (
            pool.groupby("stratum")
            .size()
            .rename("stratum_population")
        )

        sampled_counts = (
            sample_meta.groupby("stratum")
            .size()
            .rename("stratum_sampled")
        )

        sample_meta = (
            sample_meta
            .merge(
                population_counts,
                on="stratum"
            )
            .merge(
                sampled_counts,
                on="stratum"
            )
        )

        sample_meta["sampling_weight"] = (
            sample_meta["stratum_population"]
            / sample_meta["stratum_sampled"]
        )

        # Technical sampling information stored separately.
        atomic_csv(
            sample_meta,
            EXTERNAL_RUN
            / (
                "WoW_new_stratified_"
                "sampling_design_PRIVATE.csv"
            )
        )

        # Blind reviewers receive only ID and post text.
        blind_sample = (
            sample_meta[["post_id"]]
            .merge(
                source[["post_id", "full_text"]],
                on="post_id",
                validate="one_to_one"
            )
        )

        save_blind_once(
            make_blind_review(
                blind_sample,
                three_reviewers=True
            ),
            EXTERNAL_RUN
            / (
                "WoW_new_stratified_400_"
                "BLIND_REVIEW_PRIVATE.csv"
            )
        )

        atomic_csv(
            pool[["post_id", "stratum"]],
            EXTERNAL_RUN
            / (
                "WoW_candidate_"
                "sampling_frame_PRIVATE.csv"
            )
        )

        print(
            "WoW stratified review sample:",
            len(blind_sample)
        )


# ============================================================
# 8. SAVE SUMMARY COUNTS
# ============================================================

selection_summary = pd.DataFrame(
    selection_rows
)

assert len(selection_summary) == 16

atomic_csv(
    selection_summary,
    EXTERNAL_RUN
    / "external_model_family_selection_counts.csv"
)


# Human coding decisions are maintained in separately reviewed research records.
# This computational cell produces selection frames; it does not claim
# that a blank review template is evidence of completed assessment.

# ============================================================
# 10. FINAL CHECKS
# ============================================================

assert (
    EXTERNAL_RUN
    / "external_model_family_selection_counts.csv"
).is_file()

for corpus in ("DDO", "WoW"):

    assert (
        EXTERNAL_RUN
        / (
            f"{corpus}_58_vs_18v2_"
            "selection_membership_PRIVATE.csv"
        )
    ).is_file()

    assert (
        EXTERNAL_RUN
        / (
            f"{corpus}_new_union_candidates_"
            "BLIND_MANUAL_REVIEW_PRIVATE.csv"
        )
    ).is_file()

print()
print("=" * 65)
print("CELL 25 PASS")
print("=" * 65)
print("External candidate consensus completed.")
print("Historical candidate overlap completed.")
print("Blinded human-review files created or preserved.")
print("This cell exports candidates for separately documented human assessment.")
print("Output folder:", EXTERNAL_RUN)
print("=" * 65)

print()
print(selection_summary.to_string(index=False))


In [ ]:

# ============================================================
# CELL 25A — DIAGNOSTIC FIGURES (NOT JOURNAL FIGURE MASTER FILES)
# ============================================================
# Uses saved OOF predictions; no model training.
# External charts show candidate counts, not clinical outcomes.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

figure_dir = (
    MANUSCRIPT
    / "figures_FINAL_DUAL_58_18_v6_NO415"
)

figure_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. HUMAN ADJUDICATION COUNTS
# ============================================================

names = [
    "Initial reviewed",
    "Initial agreements",
    "VAS adjudications",
    "Final Include",
    "Final Exclude",
    "Final Uncertain"
]

counts = [365, 330, 35, 58, 300, 7]

assert len(labels) == 365
assert (
    len(positives),
    len(negatives),
    len(uncertain_reviewed)
) == (58, 300, 7)

fig, ax = plt.subplots(figsize=(8, 4))

ax.barh(
    names[::-1],
    counts[::-1]
)

ax.set_xlabel("Number of posts")
ax.set_title("Human-review and adjudication counts")

for index, value in enumerate(counts[::-1]):
    ax.text(
        value + 2,
        index,
        str(value),
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    max(counts) * 1.17
)

fig.tight_layout()

fig.savefig(
    figure_dir / "figure_adjudication_counts.png",
    dpi=220
)

plt.close(fig)


# ============================================================
# 2. RQ1 — PRETRAINED MENTALBERT PCA PLOTS
# ============================================================

rq1_conditions_to_plot = [
    "random_equal58",
    "five_quantile_char_length",
    "tight_one_to_one_length",
    "strict18_vs_18_tight_matched"
]

for condition in rq1_conditions_to_plot:

    assert condition in RQ1_CONDITIONS

    view = RQ1_CONDITIONS[condition]

    indices = np.asarray(
        [
            lookup[str(post_id)]
            for post_id in view.post_id
        ],
        dtype=int
    )

    X = np.asarray(
        embeddings[indices],
        dtype=np.float32
    )

    positive = (
        view.label.to_numpy(int) == 1
    )

    assert positive.any()
    assert (~positive).any()

    coordinates = PCA(
        n_components=2,
        random_state=42
    ).fit_transform(X)

    fig, ax = plt.subplots(figsize=(6.5, 5))

    ax.scatter(
        coordinates[~positive, 0],
        coordinates[~positive, 1],
        s=25,
        alpha=0.70,
        label="Comparison posts"
    )

    ax.scatter(
        coordinates[positive, 0],
        coordinates[positive, 1],
        s=25,
        alpha=0.70,
        label="Self-disclosure posts"
    )

    ax.set_xlabel("Principal component 1")
    ax.set_ylabel("Principal component 2")

    ax.set_title(
        "MentalBERT PCA — "
        + condition.replace("_", " ")
    )

    ax.legend(fontsize=8)

    fig.tight_layout()

    fig.savefig(
        figure_dir
        / f"figure_RQ1_{condition}_PCA.png",
        dpi=220
    )

    plt.close(fig)


# ============================================================
# 3. RQ2 — ROC AND PRECISION–RECALL CURVES
# ============================================================

tasks_to_plot = [
    "rq2_primary_58v116",
    "rq2_primary_58v116_hardneg",
    "rq2_candidate_58v300",
    "rq2_strict18v36",
    "rq2_strict18v36_hardneg",
    "rq2_strict18v300"
]


def aligned_oof_scores(frame, predictions):
    """
    Align predictions by post ID rather than relying
    solely on CSV row order.
    """
    assert frame.post_id.is_unique
    assert predictions.post_id.is_unique
    assert set(frame.post_id) == set(predictions.post_id)

    aligned = (
        predictions.set_index("post_id")
        .loc[frame.post_id]
        .reset_index()
    )

    if "label" in aligned.columns:
        assert np.array_equal(
            aligned.label.to_numpy(int),
            frame.label.to_numpy(int)
        )

    scores = aligned.score.to_numpy(float)

    assert np.isfinite(scores).all()
    assert (
        (scores >= 0)
        & (scores <= 1)
    ).all()

    return scores


for task in tasks_to_plot:

    frame = DATASETS[task]

    y = frame.label.to_numpy(int)

    assert set(np.unique(y)) == {0, 1}

    curves = []

    for baseline_name in (
        "tfidf_raw",
        "tfidf_masked"
    ):

        baseline_predictions = base_slice(
            task,
            baseline_name
        )

        curves.append((
            baseline_name,
            aligned_oof_scores(
                frame,
                baseline_predictions
            )
        ))

    pretrained_predictions = pretrained_slice(task)

    curves.append((
        "Pretrained MentalBERT",
        aligned_oof_scores(
            frame,
            pretrained_predictions
        )
    ))

    if RUN_FINE_TUNING:

        for representation in FT_REPS:

            ensemble = ENSEMBLE[
                (task, representation)
            ]

            curves.append((
                f"Fine-tuned {representation} (3-seed OOF mean)",
                aligned_oof_scores(
                    frame,
                    ensemble
                )
            ))

    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 4.8)
    )

    for model_name, probabilities in curves:

        fpr, tpr, _ = roc_curve(
            y,
            probabilities
        )

        precision, recall, _ = (
            precision_recall_curve(
                y,
                probabilities
            )
        )

        auc = roc_auc_score(
            y,
            probabilities
        )

        ap = average_precision_score(
            y,
            probabilities
        )

        axes[0].plot(
            fpr,
            tpr,
            label=f"{model_name}: AUC {auc:.3f}"
        )

        axes[1].plot(
            recall,
            precision,
            label=f"{model_name}: AP {ap:.3f}"
        )

    axes[0].plot(
        [0, 1],
        [0, 1],
        linestyle=":",
        linewidth=1
    )

    axes[0].set(
        title="ROC",
        xlabel="False positive rate",
        ylabel="True positive rate"
    )

    axes[1].set(
        title="Precision–recall",
        xlabel="Recall",
        ylabel="Precision"
    )

    for ax in axes:
        ax.legend(fontsize=7)

    fig.suptitle(
        "Within-corpus grouped-CV — " + task
    )

    fig.tight_layout()

    fig.savefig(
        figure_dir
        / f"figure_RQ2_{task}_ROC_PR.png",
        dpi=220
    )

    plt.close(fig)


# ============================================================
# 4. EXTERNAL CANDIDATE COUNTS
# ============================================================

external_counts_path = (
    EXTERNAL_RUN
    / "external_model_family_selection_counts.csv"
)

if external_counts_path.is_file():

    external_counts = pd.read_csv(
        external_counts_path
    )

    assert len(external_counts) == 16

    for corpus in ("DDO", "WoW"):

        subset = external_counts.loc[
            (external_counts.corpus == corpus)
            & (
                external_counts.threshold
                .round(2)
                .eq(0.60)
            )
        ].copy()

        subset["model"] = (
            subset.cohort.astype(str)
            + " / "
            + subset.representation.astype(str)
        )

        fig, ax = plt.subplots(
            figsize=(9, 4.5)
        )

        ax.barh(
            subset["model"][::-1],
            subset["candidates"][::-1]
        )

        ax.set_xlabel(
            "Posts selected by >=3/4 model families"
        )

        ax.set_title(
            f"{corpus}: candidates at threshold 0.60"
        )

        fig.tight_layout()

        fig.savefig(
            figure_dir
            / f"figure_external_{corpus}_candidate_counts.png",
            dpi=220
        )

        plt.close(fig)


print()
print("CELL 25A PASS")
print("Figures saved to:", figure_dir)
print("External counts are unvalidated candidate selections.")
